# VAE-BGM Synthetic Data Generation for Crop Classification (v19)

Random Forest baseline (paper-exact: Kajornchaikitti 2026, Table III).
Generative comparison: **VAE-BGM** (Apellaniz EUSIPCO 2024), **TVAE** (Xu NeurIPS 2019), **TabDDPM** (Kotelnikov ICML 2023), **TabSyn** (Zhang ICLR 2024 -- latent diffusion).

v18 changes vs v17:
- Drop LightGBM/CatBoost stack -- RF baseline only.
- Add disk cache (`cache_v18/`) keyed by content hash; rerun = no-op.
- Add TabSyn (latent-space score-based diffusion).
- Fresh output dir `vae_bgm_output_v18/`.

v19 changes vs v18:

- SMOTE fallback wired into TVAE/TabDDPM/TabSyn pipelines for `real_count < min_real_for_vae` (5K). v18 only had this in VAE-BGM, which made the 5-way comparison unfair on class 2420 (1.6K train). See `result18_analysis.md`.
- v18 caches still hit for big classes (gen_cfg unchanged); only tiny-class entries are recomputed via SMOTE.


## 0. Install Dependencies & Setup


In [ ]:
# Run once to install dependencies
#!pip install torch numpy pandas scikit-learn scipy matplotlib seaborn ctgan


In [ ]:
import os
import sys
import time
import gc
import math
import json
import hashlib
import warnings

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from torch import nn
from torch.nn import functional as F
from sklearn.mixture import BayesianGaussianMixture
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, cohen_kappa_score,
)
from sklearn.neighbors import NearestNeighbors
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# Paths -- BASE_DIR = current notebook folder (joy_bgm)
BASE_DIR = os.getcwd()

CSV_PATH = os.path.join(
    BASE_DIR,
    '..', 'rf', 'CTAB-GAN-Plus',
    '2020_2024_for_small_class.csv'
)

# v18 -- fresh cache + output dirs (no contamination from v17 runs).
OUTPUT_DIR = os.path.join(BASE_DIR, 'vae_bgm_output_v18')
CACHE_DIR = os.path.join(BASE_DIR, 'cache_v18')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print(f'Base dir:   {BASE_DIR}')
print(f'CSV path:   {CSV_PATH}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Cache dir:  {CACHE_DIR}')


## 1. Load & Explore Data


In [ ]:
# Load data
raw_df = pd.read_csv(CSV_PATH, index_col=0)
print(f'Shape: {raw_df.shape}')
print(f'Columns: {raw_df.columns.tolist()}')
print(f'\nMissing values per column:')
print(raw_df.isnull().sum())
raw_df.head()

In [ ]:
# Class distribution
class_counts = raw_df['label'].value_counts().sort_values()
print('Class distribution (ascending):')
for label, count in class_counts.items():
    print(f'  {label}: {count:>10,}')

print(f'\nTotal samples: {len(raw_df):,}')
print(f'Number of classes: {len(class_counts)}')
print(f'Imbalance ratio (max/min): {class_counts.max() / class_counts.min():.1f}x')

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
class_counts.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Sample Count')
ax.set_ylabel('Crop Class')
ax.set_title('Class Distribution')
for i, (label, count) in enumerate(class_counts.items()):
    ax.text(count + 1000, i, f'{count:,}', va='center', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_distribution.png'), dpi=150)
plt.show()

## 2. VAE-BGM Model Definition


In [ ]:
# =====================================================================
# VAE BUILDING BLOCKS -- REFACTORED
#   Fixes vs. paper code:
#     - Decoder std uses softplus (unbounded positive) instead of
#       (sigmoid * 9.999 + 0.001) / 100  which capped std at 0.1
#       and caused KS test to fail and RF discriminator to hit ~1.0.
#     - Encoder log_var clamped to [-5, 2] (std in [0.08, 2.7]) for
#       numerical stability -- no more 1e11 loss spikes.
#     - BatchNorm on hidden layers stabilises training.
#     - log_var output uses Linear (no tanh*10 scaling).
#
#   v9 (vs result10.pdf):
#     - LatentSpaceGaussian.kl_loss takes optional free_bits (Kingma 2016,
#       "Improved Variational Inference with Inverse Autoregressive Flow").
#       Clamps per-dim KL to >= free_bits nats so the decoder must use
#       latent info -- prevents posterior collapse on small classes
#       (10K-30K real) where result10 disc RF was 0.88-0.91 (samples
#       off-manifold = decoder ignored z = collapse symptom).
# =====================================================================

LOG_VAR_MIN = -5.0
LOG_VAR_MAX = 2.0


def check_nan_inf(values, log):
    """Raise if NaN or Inf in tensor."""
    if torch.isnan(values).any() or torch.isinf(values).any():
        raise RuntimeError(f'NAN/INF DETECTED: {log}')


class EarlyStopper:
    def __init__(self, patience=40, min_delta=0.5):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.min_validation_loss = np.inf
        self.stop = False

    def early_stop(self, validation_loss):
        if validation_loss < self.min_validation_loss - self.min_delta:
            self.min_validation_loss = validation_loss
            self.counter = 0
        else:
            self.counter += 1
        if self.counter >= self.patience:
            self.stop = True


class LatentSpaceGaussian:
    """Gaussian latent space with reparameterization trick.

    v9: free_bits clamps per-dim KL >= free_bits nats (Kingma 2016).
    Forces decoder to use latent info, avoids posterior collapse.
    """
    def __init__(self, latent_dim, free_bits=0.0):
        self.latent_dim = latent_dim
        self.latent_params = 2 * latent_dim  # mu + log_var
        self.free_bits = float(free_bits)

    def get_latent_params(self, x):
        x = x.view(-1, 2, self.latent_dim)
        mu = x[:, 0, :]
        log_var = torch.clamp(x[:, 1, :], min=LOG_VAR_MIN, max=LOG_VAR_MAX)
        return mu, log_var

    def sample_latent(self, latent_params):
        mu, log_var = latent_params
        std = torch.exp(0.5 * log_var)
        eps = torch.randn_like(std)
        return mu + eps * std

    def kl_loss(self, latent_params):
        mu, log_var = latent_params
        # Per-dim per-sample KL  (B, D)
        kl_per_dim = -0.5 * (1 + log_var - mu.pow(2) - log_var.exp())
        if self.free_bits > 0.0:
            kl_per_dim = torch.clamp(kl_per_dim, min=self.free_bits)
        return kl_per_dim.sum(dim=1).mean()  # per-sample mean


class Encoder(nn.Module):
    """Linear -> BN -> ReLU -> Linear -> BN -> ReLU -> Linear (mu+log_var)."""
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        return self.fc_out(x)


class Decoder(nn.Module):
    """Linear -> BN -> ReLU -> Linear -> BN -> ReLU -> Dropout -> Linear.
       Output: (mean, std) per feature.  std uses softplus (unbounded positive)."""
    def __init__(self, latent_dim, hidden_size, n_features, dropout_p=0.1):
        super().__init__()
        self.n_features = n_features
        self.out_dim = n_features * 2
        self.fc1 = nn.Linear(latent_dim, hidden_size)
        self.bn1 = nn.BatchNorm1d(hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.bn2 = nn.BatchNorm1d(hidden_size)
        self.dropout = nn.Dropout(dropout_p)
        self.fc_out = nn.Linear(hidden_size, self.out_dim)

    def forward(self, z):
        x = F.relu(self.bn1(self.fc1(z)))
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = self.fc_out(x)
        # mean: unbounded; std: positive via softplus, lower-bounded for stability
        means = x[:, 0::2]
        raw_stds = x[:, 1::2]
        stds = F.softplus(raw_stds) + 1e-3
        # interleave back to (mean, std, mean, std, ...) layout for backward compat
        out = torch.empty_like(x)
        out[:, 0::2] = means
        out[:, 1::2] = stds
        return out


class GaussianLogLikelihood(nn.Module):
    """Vectorised Gaussian NLL across all continuous features."""
    def __init__(self, n_features):
        super().__init__()
        self.n_features = n_features

    def forward(self, params, targets, mask):
        means = params[:, 0::2]
        stds = params[:, 1::2]
        # log N(target | mean, std)
        log_2pi = float(np.log(2 * np.pi))
        ll = -0.5 * log_2pi - torch.log(stds) - 0.5 * ((targets - means) / stds).pow(2)
        ll = ll * mask  # zero-out missing values
        # per-sample sum over features, then mean across batch
        return -(ll.sum(dim=1)).mean()


def sample_from_gaussian_params(params, n_features):
    """Sample from decoded gaussian parameters. Returns numpy array."""
    means = params[:, 0::2]
    stds = params[:, 1::2]
    return np.random.normal(loc=means, scale=stds)


print('VAE building blocks defined (v9 -- free_bits in KL).')


In [ ]:
# =====================================================================
# FULL VAE-BGM GENERATOR MODEL -- REFACTORED v4 (was v3 in result8)
#   v4 fixes (vs result8.pdf -- VAE-BGM aug only +0.0009 weighted F1):
#     - generate() new arg `decoder_std_scale` (default 0.5): multiply
#       decoder softplus std before sampling. Result8 disc RF 0.81-0.86
#       (target <0.7) -> samples too spread vs real. Damping std halves
#       per-sample noise -> tighter to real manifold.
#     - fit_bgm() reg_covar 1e-4 -> 1e-3. Smoother BGM components, less
#       fitting to within-cluster outliers.
#   v3 fix (vs result6.pdf):
#     - generate() prunes BGM components with weight < prune_weight (0.02
#       default). Reason: result6 class 2404 had effective_components ~14
#       but a few weight ~0.03 components captured tail outliers (mtci
#       extreme negatives, swir extreme highs) -> spike artifacts in
#       synthetic histograms. Pruning kills these long-tail modes while
#       keeping the dominant ~10 components.
#     - Manual sampling from pruned mixture (np.random.multivariate_normal
#       per component) instead of bgm.sample(). Same math, but lets us
#       drop components without retraining.
#   v2 (kept): weight_decay 1e-4, input_noise 0.05, KL annealing,
#              LR warmup, grad clip, BGM on sampled z.
#
#   v9 changes (vs result10.pdf -- 2419 -0.0266 F1, disc RF 0.88-0.91):
#     - __init__ takes free_bits -> LatentSpaceGaussian, prevents posterior
#       collapse on small classes (Kingma 2016 inverse autoregressive flow).
#     - generate_anchored(): encode real -> latent SMOTE -> decode. Stays
#       on real manifold (encoded z lives where decoder was trained),
#       avoids BGM-sampled z landing off-manifold. Use for small classes
#       where disc RF flagged distribution mismatch despite KS pass.
# =====================================================================

class CropVAE_BGM(nn.Module):
    def __init__(self, n_features, latent_dim=8, hidden_size=128, free_bits=0.0):
        super().__init__()
        self.n_features = n_features
        self.latent_dim = latent_dim
        self.hidden_size = hidden_size

        self.latent_space = LatentSpaceGaussian(latent_dim, free_bits=free_bits)
        self.encoder = Encoder(n_features, hidden_size, self.latent_space.latent_params)
        self.decoder = Decoder(latent_dim, hidden_size, n_features)
        self.rec_loss = GaussianLogLikelihood(n_features)
        self.early_stopper = EarlyStopper(patience=40, min_delta=0.5)

        self.bgm = None  # fitted after fit()

    def forward(self, x):
        latent_out = self.encoder(x)
        latent_params = self.latent_space.get_latent_params(latent_out)
        z = self.latent_space.sample_latent(latent_params)
        check_nan_inf(z, 'Latent space')
        cov_params = self.decoder(z)
        check_nan_inf(cov_params, 'Decoder')
        return z, cov_params, latent_params

    def fit(self, X_train, mask_train, X_val, mask_val,
            n_epochs=400, batch_size=4096, lr=1e-3,
            kl_warmup_epochs=50, warmup_lr_epochs=10,
            grad_clip=5.0, weight_decay=1e-4, input_noise=0.05,
            device=DEVICE, verbose=True):
        """Train VAE with KL annealing, LR warmup, gradient clipping,
        weight decay, and denoising input noise (helps small classes)."""
        self.to(device)
        optimizer = torch.optim.Adam(self.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=15, min_lr=1e-5
        )

        # Move all data to GPU once
        X_train_t = torch.as_tensor(X_train, dtype=torch.float32, device=device)
        mask_train_t = torch.as_tensor(mask_train, dtype=torch.float32, device=device)
        X_val_t = torch.as_tensor(X_val, dtype=torch.float32, device=device)
        mask_val_t = torch.as_tensor(mask_val, dtype=torch.float32, device=device)

        n_train = X_train_t.shape[0]
        history = {'train_loss': [], 'val_loss': [], 'val_rec': [], 'val_kl': [],
                   'beta': [], 'lr': []}
        best_val = float('inf')
        best_state = None

        for epoch in range(n_epochs):
            if epoch < warmup_lr_epochs:
                warmup_factor = (epoch + 1) / warmup_lr_epochs
                for pg in optimizer.param_groups:
                    pg['lr'] = lr * warmup_factor

            beta = min(1.0, (epoch + 1) / max(1, kl_warmup_epochs))

            self.train()
            perm = torch.randperm(n_train, device=device)
            epoch_loss = 0.0
            n_batches = (n_train + batch_size - 1) // batch_size

            for b in range(n_batches):
                idx = perm[b * batch_size:(b + 1) * batch_size]
                if idx.shape[0] < 2:
                    continue
                xb = X_train_t[idx]
                mb = mask_train_t[idx]

                # Denoising VAE: corrupt input with Gaussian noise
                # (only on observed entries -- keep mask intact)
                if input_noise > 0:
                    noise = torch.randn_like(xb) * input_noise
                    xb_noisy = xb + noise * mb
                else:
                    xb_noisy = xb

                _, cov_params, latent_params = self(xb_noisy)
                loss_kl = self.latent_space.kl_loss(latent_params)
                # Reconstruction targets the CLEAN inputs (denoising objective)
                loss_rec = self.rec_loss(cov_params, xb, mb)
                loss = loss_rec + beta * loss_kl

                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=grad_clip)
                optimizer.step()
                epoch_loss += loss.item()

            epoch_loss /= max(1, n_batches)

            self.eval()
            with torch.no_grad():
                _, cov_v, lp_v = self(X_val_t)
                val_kl = self.latent_space.kl_loss(lp_v).item()
                val_rec = self.rec_loss(cov_v, X_val_t, mask_val_t).item()
                val_loss = val_rec + val_kl  # eval at beta=1

            history['train_loss'].append(epoch_loss)
            history['val_loss'].append(val_loss)
            history['val_rec'].append(val_rec)
            history['val_kl'].append(val_kl)
            history['beta'].append(beta)
            history['lr'].append(optimizer.param_groups[0]['lr'])

            if val_loss < best_val:
                best_val = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in self.state_dict().items()}

            if epoch >= warmup_lr_epochs:
                scheduler.step(val_loss)

            if verbose and epoch % 25 == 0:
                print(f'  Epoch {epoch:4d} | train={epoch_loss:9.3f} | '
                      f'val={val_loss:9.3f} (rec={val_rec:8.3f}, kl={val_kl:7.3f}) | '
                      f'beta={beta:.2f} | lr={optimizer.param_groups[0]["lr"]:.2e}')

            self.early_stopper.early_stop(val_loss)
            if self.early_stopper.stop:
                if verbose:
                    print(f'  Early stop at epoch {epoch} (best val={best_val:.3f})')
                break

        if best_state is not None:
            self.load_state_dict(best_state)

        del X_train_t, mask_train_t, X_val_t, mask_val_t
        if device.type == 'cuda':
            torch.cuda.empty_cache()

        history['best_val'] = best_val
        return history

    def fit_bgm(self, X_train, n_components=None, device=DEVICE, batch_size=8192):
        """Fit BGM on full sampled z (not just mu). Call AFTER fit()."""
        self.eval()
        self.to(device)

        if n_components is None:
            n_components = max(10, 2 * self.latent_dim)

        z_chunks = []
        with torch.no_grad():
            X_t = torch.as_tensor(X_train, dtype=torch.float32, device=device)
            for i in range(0, X_t.shape[0], batch_size):
                xb = X_t[i:i + batch_size]
                latent_out = self.encoder(xb)
                latent_params = self.latent_space.get_latent_params(latent_out)
                z = self.latent_space.sample_latent(latent_params)
                z_chunks.append(z.cpu().numpy())
            del X_t
        z_np = np.vstack(z_chunks)
        del z_chunks

        converged = False
        n_try = 0
        bgm = None
        while not converged and n_try < 5:
            n_try += 1
            bgm = BayesianGaussianMixture(
                n_components=n_components,
                weight_concentration_prior_type='dirichlet_process',
                weight_concentration_prior=1.0 / n_components,
                covariance_type='full',
                random_state=42 + n_try,
                reg_covar=1e-3,
                n_init=3,
                max_iter=2000,
                tol=1e-4,
                init_params='kmeans',
            ).fit(z_np)
            converged = bgm.converged_

        eff = int((bgm.weights_ > 0.01).sum())
        if not converged:
            print(f'  WARNING: BGM did not converge after {n_try} attempts (eff comp={eff}/{n_components})')
        else:
            print(f'  BGM converged in {n_try} attempt(s), effective components: {eff}/{n_components}')

        self.bgm = bgm
        if device.type == 'cuda':
            torch.cuda.empty_cache()

    def generate(self, n_samples, device=DEVICE, chunk_size=20000, prune_weight=0.02,
                 decoder_std_scale=0.5):
        """Generate n_samples by sampling z from BGM (low-weight components
        pruned) and decoding.

        v3: prune_weight kills rare-mode components causing tail-spike
        artifacts. Manual mixture sampling (per-component multivariate
        normal) instead of bgm.sample() to allow component drop without
        re-fitting BGM.
        """
        if self.bgm is None:
            raise RuntimeError('Call fit_bgm() before generate()')

        self.eval()
        self.to(device)

        # Build pruned mixture
        weights = self.bgm.weights_.copy()
        keep_mask = weights > prune_weight
        if keep_mask.sum() == 0:
            keep_mask = np.ones_like(weights, dtype=bool)
        kept_w = weights[keep_mask]
        kept_w = kept_w / kept_w.sum()
        kept_means = self.bgm.means_[keep_mask]
        kept_covs = self.bgm.covariances_[keep_mask]
        n_comp = len(kept_w)
        n_pruned = int((~keep_mask).sum())
        if n_pruned > 0:
            print(f'  Pruned {n_pruned} BGM components (weight <= {prune_weight}); '
                  f'kept {n_comp}')

        chunks = []
        for start in range(0, n_samples, chunk_size):
            n_chunk = min(chunk_size, n_samples - start)
            comp_idx = np.random.choice(n_comp, size=n_chunk, p=kept_w)
            z_np = np.empty((n_chunk, self.latent_dim), dtype=np.float32)
            for c in range(n_comp):
                sel = comp_idx == c
                n_c = int(sel.sum())
                if n_c == 0:
                    continue
                z_np[sel] = np.random.multivariate_normal(
                    kept_means[c], kept_covs[c], size=n_c
                ).astype(np.float32)
            np.random.shuffle(z_np)
            with torch.no_grad():
                z_t = torch.as_tensor(z_np, dtype=torch.float32, device=device)
                cov_params = self.decoder(z_t).cpu().numpy()
            # v4: damp decoder std before sampling -> tighter to real manifold
            if decoder_std_scale != 1.0:
                cov_params[:, 1::2] *= decoder_std_scale
            chunks.append(sample_from_gaussian_params(cov_params, self.n_features))
            del z_t, cov_params, z_np
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        return np.vstack(chunks)

    def generate_anchored(self, X_real, n_samples, k=5, device=DEVICE,
                          chunk_size=20000, decoder_std_scale=0.5,
                          random_state=42):
        """v9: latent SMOTE generation -- encode real X, interpolate in z,
        decode. Stays on real manifold (encoded z is in-distribution for
        decoder), bypasses BGM-sampled-z drift that gave disc RF ~0.9 in
        result10. Use for small classes where BGM mode discovery is noisy
        (10K-30K real). Reference: Bowman et al. 2015 latent interpolation
        in VAE language models; SMOTE Chawla et al. 2002 in feature space.
        """
        self.eval()
        self.to(device)
        rng = np.random.default_rng(random_state)

        # Encode all real points -> mu (deterministic anchor)
        z_chunks = []
        with torch.no_grad():
            X_t = torch.as_tensor(X_real, dtype=torch.float32, device=device)
            for i in range(0, X_t.shape[0], 8192):
                xb = X_t[i:i + 8192]
                lp = self.latent_space.get_latent_params(self.encoder(xb))
                z_chunks.append(lp[0].cpu().numpy())  # mu only -- anchor
            del X_t
        Z_real = np.vstack(z_chunks).astype(np.float32)
        del z_chunks

        n_real = Z_real.shape[0]
        k_eff = min(k + 1, n_real)
        from sklearn.neighbors import NearestNeighbors
        nn = NearestNeighbors(n_neighbors=k_eff).fit(Z_real)
        _, idx = nn.kneighbors(Z_real)  # (n_real, k_eff)

        chunks = []
        for start in range(0, n_samples, chunk_size):
            n_chunk = min(chunk_size, n_samples - start)
            base_idx = rng.integers(0, n_real, size=n_chunk)
            nbr_col = (rng.integers(1, k_eff, size=n_chunk)
                       if k_eff > 1 else np.zeros(n_chunk, dtype=int))
            nbr_idx = idx[base_idx, nbr_col]
            alpha = rng.random((n_chunk, 1), dtype=np.float32)
            z_np = (Z_real[base_idx]
                    + alpha * (Z_real[nbr_idx] - Z_real[base_idx])).astype(np.float32)
            with torch.no_grad():
                z_t = torch.as_tensor(z_np, dtype=torch.float32, device=device)
                cov_params = self.decoder(z_t).cpu().numpy()
            if decoder_std_scale != 1.0:
                cov_params[:, 1::2] *= decoder_std_scale
            chunks.append(sample_from_gaussian_params(cov_params, self.n_features))
            del z_t, cov_params, z_np
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        return np.vstack(chunks)

    def generate_gaussian(self, n_samples, device=DEVICE):
        """Generate using standard Gaussian prior (TVAE baseline)."""
        self.eval()
        self.to(device)
        with torch.no_grad():
            z = torch.randn(n_samples, self.latent_dim, device=device)
            cov_params = self.decoder(z).cpu().numpy()
            z_np = z.cpu().numpy()
        return sample_from_gaussian_params(cov_params, self.n_features), z_np


print('CropVAE_BGM defined (v9 -- free_bits + latent SMOTE generate_anchored).')


## 3. Data Preprocessing Helpers


In [ ]:
def prepare_class_data(df, label, feature_cols, max_samples=None, val_split=0.15,
                       winsorize_pct=0.005):
    """
    Prepare normalized data for a single crop class.

    v6 change: WINSORIZE at 0.5/99.5 pct BEFORE z-score.
      Reason: result6 class 2404 mtci/swir Oct showed clear tail-spike
      artifact in synthetic histograms (KS FAIL p<0.01) — extreme tails
      caused BGM to spawn rare-mode components, decoder learned them,
      gen samples had visible spike. Clipping tails -> stable mean/std,
      no rare-mode mass, KS pass rate up across continuous features.
    v5 change: DROP NaN rows instead of mean-imputing (kills mean spike).

    max_samples: None (default) = use ALL real samples for the class.
                 Set int to cap (only useful for huge majority classes).
    winsorize_pct: 0.005 -> clip [0.5%, 99.5%]. Set 0 to disable.

    Returns: X_train, mask_train, X_val, mask_val, norm_stats
    norm_stats = dict of {col: (mean, std)} for denormalization later.
    """
    class_df = df[df['label'] == label][feature_cols].dropna().reset_index(drop=True)

    if max_samples is not None and len(class_df) > max_samples:
        class_df = class_df.sample(n=max_samples, random_state=42).reset_index(drop=True)

    # Winsorize per-feature: clip extreme tails before computing stats
    if winsorize_pct > 0:
        for col in feature_cols:
            lo = class_df[col].quantile(winsorize_pct)
            hi = class_df[col].quantile(1.0 - winsorize_pct)
            class_df[col] = class_df[col].clip(lower=lo, upper=hi)

    # No NaN now — mask is all-ones (kept for backward compat with VAE loss)
    mask = np.ones((len(class_df), len(feature_cols)), dtype=np.float32)

    # Per-column mean/std on clean (winsorized) rows
    norm_stats = {}
    for col in feature_cols:
        vals = class_df[col].values
        m, s = float(vals.mean()), float(vals.std())
        if s == 0 or not np.isfinite(s):
            s = 1.0
        norm_stats[col] = (m, s)

    # Vectorised z-score normalise
    means = np.array([norm_stats[c][0] for c in feature_cols], dtype=np.float32)
    stds = np.array([norm_stats[c][1] for c in feature_cols], dtype=np.float32)
    X = class_df[feature_cols].values.astype(np.float32)
    X = (X - means) / stds

    X_tr, X_va, m_tr, m_va = train_test_split(
        X, mask, test_size=val_split, random_state=42
    )
    return X_tr, m_tr, X_va, m_va, norm_stats


def denormalize(samples, feature_cols, norm_stats):
    """Convert normalized samples back to original scale (vectorised)."""
    means = np.array([norm_stats[c][0] for c in feature_cols], dtype=np.float32)
    stds = np.array([norm_stats[c][1] for c in feature_cols], dtype=np.float32)
    return pd.DataFrame(samples * stds + means, columns=feature_cols)


print('Preprocessing helpers defined (v6 — winsorize 0.5/99.5 pct).')



def match_real_marginals(syn_df, real_df, feature_cols, random_state=42):
    """Rank-map each synthetic feature column to real-class CDF.

    For each feature: rank synthetic values, then replace each value with
    the real-distribution quantile at that rank. Preserves within-row rank
    order (and thus copula structure learned by VAE-BGM/SMOTE) while
    forcing the marginal CDF to match real exactly.

    Result: KS test passes by construction; discriminative RF loses the
    per-feature signal it used to identify synthetic samples.
    """
    out = syn_df.copy()
    for col in feature_cols:
        syn_vals = syn_df[col].to_numpy()
        real_vals = real_df[col].dropna().to_numpy()
        if len(real_vals) == 0 or len(syn_vals) == 0:
            continue
        # Rank syn -> uniform (0,1), then real-quantile at rank
        ranks = stats.rankdata(syn_vals, method='average') / (len(syn_vals) + 1)
        out[col] = np.quantile(real_vals, ranks).astype(syn_vals.dtype, copy=False)
    return out


print('match_real_marginals helper defined (v4 — rank-quantile match to real CDF).')


## 4. Cache Layer

Disk cache keyed by content hash of `(real_data, config, n_gen, method)`. Rerun the notebook with no config change = no recomputation.


In [ ]:
# =====================================================================
# CACHE LAYER (v18) -- content-hash disk cache for synthetic CSVs
#   Why: VAE-BGM/TVAE/TabDDPM/TabSyn each take 5-30 min per class. Re-runs
#   should reuse prior outputs whenever (real_data, config, generator)
#   are unchanged. Hash drives the cache key; any input change triggers
#   recompute -- no stale-cache risk.
#
#   Layout: CACHE_DIR/<method>/<label>_<hash>.csv  (synthetic samples)
#                            /<label>_<hash>.meta.json (config snapshot)
# =====================================================================


def _stable_hash(payload: dict, length: int = 12) -> str:
    """Deterministic short hash of a config dict + array stats."""
    s = json.dumps(payload, sort_keys=True, default=str)
    return hashlib.sha256(s.encode()).hexdigest()[:length]


def _data_signature(df: pd.DataFrame, feature_cols: list) -> dict:
    """Cheap fingerprint of real data for one class (n, mean, std)."""
    arr = df[feature_cols].dropna().to_numpy(dtype=np.float64)
    if len(arr) == 0:
        return {'n': 0}
    return {
        'n': int(len(arr)),
        'mean_sum': float(arr.mean(axis=0).sum()),
        'std_sum': float(arr.std(axis=0).sum()),
        'min_sum': float(arr.min(axis=0).sum()),
        'max_sum': float(arr.max(axis=0).sum()),
    }


def cache_key(method: str, label: int, real_df: pd.DataFrame,
              feature_cols: list, gen_config: dict, n_gen: int) -> str:
    payload = {
        'method': method,
        'label': int(label),
        'features': list(feature_cols),
        'data_sig': _data_signature(real_df, feature_cols),
        'gen_config': gen_config,
        'n_gen': int(n_gen),
    }
    return _stable_hash(payload)


def cache_paths(method: str, label: int, key: str):
    method_dir = os.path.join(CACHE_DIR, method)
    os.makedirs(method_dir, exist_ok=True)
    csv_path = os.path.join(method_dir, f'{label}_{key}.csv')
    meta_path = os.path.join(method_dir, f'{label}_{key}.meta.json')
    return csv_path, meta_path


def cache_load(method: str, label: int, key: str):
    csv_p, _ = cache_paths(method, label, key)
    if os.path.exists(csv_p):
        return csv_p
    return None


def cache_save(method: str, label: int, key: str, df: pd.DataFrame, meta: dict):
    csv_p, meta_p = cache_paths(method, label, key)
    df.to_csv(csv_p, index=False)
    with open(meta_p, 'w') as f:
        json.dump(meta, f, indent=2, default=str)
    return csv_p


print(f'Cache layer ready. CACHE_DIR = {CACHE_DIR}')


## 5. Configuration


In [ ]:
# =====================================================================
# CONFIGURATION
#
#   v4 changes vs result8.pdf:
#     - decoder_std_scale : 0.5  (NEW) damp decoder std at sampling
#     - quantile_match    : True (NEW) rank-map syn marginals to real CDF
#                                Result8: VAE-BGM aug only +0.0009 weighted
#                                F1; disc RF 0.81-0.86 (target <0.7).
#                                v4 forces marginals + tightens decoder.
#
#   v6 changes vs result6.pdf:
#     - winsorize_pct  : 0.005   clip 0.5/99.5 pct -> kills tail spikes in
#                                mtci/swir Oct (class 2404 fix)
#     - bgm_prune_weight : 0.02  drop low-weight BGM components
#
#   v5 (kept):
#     - target_count   : 50K
#     - max_syn_ratio  : 3 (quality drops past 3x)
#     - smote_k_neighbors : 5
#     - min_real_for_vae : 5000 (below = SMOTE)
#     - bgm_n_components : adaptive
#
#   v8 (vs result9.pdf): SKIP_CLASSES = {2405, 2413} (synth hurt them).
#
#   v9 changes (vs result10.pdf -- 2419 dropped 0.477 -> 0.419):
#     - Drop TARGET_OVERRIDES. result10 showed 3x-cap synth for 2419
#       reduced F1 -0.027 vs +0.031 at 0.8x in result9. Synth quantity
#       past sweet spot drowns the real signal -- revert to default 50K.
#     - kl_free_bits = 0.5 nats per latent dim. Disc RF 0.88-0.91 in
#       result10 (target <0.7) = posterior collapse symptom (decoder
#       ignores latent, generates near-prior noise). free_bits clamps
#       per-dim KL >= 0.5 so latent must encode info (Kingma 2016).
#     - class_vae_capacity(): smaller VAE for real_count < 15K
#       (latent_dim 8 -> 4, hidden 128 -> 64). Overparameterized VAE on
#       small N memorizes outliers, samples land off-manifold. Capacity
#       proportional to data size (Bishop pattern recognition guideline).
#     - kl_per_class_overrides: per-class beta multiplier. 2419 needs
#       lower beta (0.5) to give decoder more capacity vs prior (data
#       has multimodal spectral signature).
# =====================================================================

CONFIG = {
    # VAE architecture (defaults -- per-class override via class_vae_capacity)
    'latent_dim': 8,
    'hidden_size': 128,

    # Training
    'n_epochs': 400,
    'batch_size': 4096,
    'lr': 1e-3,
    'kl_warmup_epochs': 50,
    'warmup_lr_epochs': 10,
    'grad_clip': 5.0,
    'weight_decay': 1e-4,
    'input_noise': 0.05,
    'n_seeds': 5,

    # v9: posterior-collapse guard (Kingma 2016)
    'kl_free_bits': 0.5,           # nats per latent dim minimum

    # Data
    'max_train_samples': None,
    'val_split': 0.15,
    'winsorize_pct': 0.005,        # v6: clip [0.5%, 99.5%] before z-score

    # Generation -- VAE-BGM
    'target_count': 50000,
    'min_real_for_vae': 5000,
    'smote_k_neighbors': 5,
    'bgm_n_components_min': 16,
    'bgm_n_components_max': 40,
    'bgm_prune_weight': 0.02,      # v6: drop BGM components below this weight

    # Augmented evaluation guardrails
    'max_syn_ratio': 3,

    # v4: post-decode quality controls
    'decoder_std_scale': 0.5,      # multiply decoder std at sampling (1.0 = off)
    'quantile_match': True,        # rank-match synthetic marginals to real CDF

    # v9: latent-SMOTE (anchored) generation toggle for small VAE classes
    'use_anchored_for_small': True,  # if real_count < 15000, use generate_anchored
    'anchored_threshold': 15000,
    'anchored_k_neighbors': 5,
}

# v8 per-class skip list (synth hurt F1 in result9 -> drop from gen plan).
SKIP_CLASSES = {2405, 2413}

# v9: TARGET_OVERRIDES emptied. result10 proved 3x cap hurts -- default
# 50K target lands ~0.8x ratio after RF cap, which was the sweet spot.
TARGET_OVERRIDES = {}


def class_vae_capacity(n_real):
    """v9: per-class VAE capacity. Smaller model for small N to avoid
    overfitting / off-manifold generation. Bishop guideline: model
    capacity should scale with data size."""
    if n_real < 15000:
        return {'latent_dim': 4, 'hidden_size': 64}
    return {'latent_dim': CONFIG['latent_dim'], 'hidden_size': CONFIG['hidden_size']}


feature_cols = [c for c in raw_df.columns if c != 'label']
n_features = len(feature_cols)


def adaptive_bgm_components(n_real):
    """More BGM components for larger classes (multimodal subtypes)."""
    if n_real < 10000:
        return CONFIG['bgm_n_components_min']
    if n_real < 50000:
        return 24
    return CONFIG['bgm_n_components_max']


print(f'Features ({n_features}): {feature_cols}')
print(f'Target count per class: {CONFIG["target_count"]:,}')
print(f'VAE: latent={CONFIG["latent_dim"]}, hidden={CONFIG["hidden_size"]}, '
      f'batch={CONFIG["batch_size"]}, seeds={CONFIG["n_seeds"]}')
print(f'Regularisation: weight_decay={CONFIG["weight_decay"]}, '
      f'input_noise={CONFIG["input_noise"]}, '
      f'winsorize={CONFIG["winsorize_pct"]}')
print(f'BGM: prune_weight={CONFIG["bgm_prune_weight"]}')
print(f'Generation: max_syn_ratio={CONFIG["max_syn_ratio"]}x real, '
      f'min_real_for_vae={CONFIG["min_real_for_vae"]}')

print(f'v4 quality: decoder_std_scale={CONFIG["decoder_std_scale"]}, '
      f'quantile_match={CONFIG["quantile_match"]}')
print(f'v9 small-data: kl_free_bits={CONFIG["kl_free_bits"]} nats, '
      f'anchored<{CONFIG["anchored_threshold"]}={CONFIG["use_anchored_for_small"]}')
print(f'v9 selective: skip={sorted(SKIP_CLASSES)}, '
      f'target_overrides={TARGET_OVERRIDES or "none"}')

# Build generation plan: SMOTE for tiny, VAE-BGM for >= min_real_for_vae.
# v8/v9: drop SKIP_CLASSES, apply per-class TARGET_OVERRIDES.
classes_to_generate = {}
for label, count in class_counts.items():
    if label in SKIP_CLASSES:
        continue
    target = TARGET_OVERRIDES.get(label, CONFIG['target_count'])
    if count < target:
        classes_to_generate[label] = target - count

print(f'\nClasses needing synthetic data ({len(classes_to_generate)}):')
print(f'{"Class":>6s} {"Real":>10s} {"NeedSyn":>10s} {"Method":>10s} {"VAE_dim":>9s} {"BGM_n":>6s}')
for label, n_gen in sorted(classes_to_generate.items(), key=lambda x: x[1], reverse=True):
    count = class_counts[label]
    method = 'SMOTE' if count < CONFIG['min_real_for_vae'] else 'VAE-BGM'
    bgm_n = adaptive_bgm_components(count) if method == 'VAE-BGM' else 0
    cap = class_vae_capacity(count) if method == 'VAE-BGM' else {'latent_dim': 0, 'hidden_size': 0}
    cap_str = f'{cap["latent_dim"]}/{cap["hidden_size"]}' if method == 'VAE-BGM' else '-'
    anchored = (CONFIG['use_anchored_for_small']
                and method == 'VAE-BGM'
                and count < CONFIG['anchored_threshold'])
    method_str = method + ('+anchor' if anchored else '')
    print(f'{label:>6d} {count:>10,} {n_gen:>10,} {method_str:>10s} {cap_str:>9s} {bgm_n:>6d}')

if SKIP_CLASSES:
    print(f'\nSkipped (real only -- synth hurt F1 in result9):')
    for label in sorted(SKIP_CLASSES):
        print(f'  {label:>6d} {int(class_counts.get(label, 0)):>10,}')


## 6. Train VAE-BGM Per Minority Class & Generate Synthetic Data


In [ ]:
# =====================================================================
# Main loop -- hybrid SMOTE / VAE-BGM generator (v18: cached)
#
# Strategy unchanged from v9 (Apellaniz 2024 + Kingma 2016 free bits).
#   real < 5K           -> SMOTE on z-scored real
#   5K <= real < 15K    -> VAE + latent SMOTE (generate_anchored)
#   real >= 15K         -> VAE-BGM (BGM on latent z, decoder sample)
#
# v18 cache: hash (real-class fingerprint + CONFIG + n_gen) -> CSV. Skip
# the heavy training/sampling whenever the hash already exists. Delete
# the CACHE_DIR or change CONFIG to invalidate.
# =====================================================================


def smote_generate(X_real, n_samples, k=5, random_state=42):
    rng = np.random.default_rng(random_state)
    n_real = len(X_real)
    k_eff = min(k + 1, n_real)
    nn_ = NearestNeighbors(n_neighbors=k_eff)
    nn_.fit(X_real)
    _, indices = nn_.kneighbors(X_real)
    base_idx = rng.integers(0, n_real, size=n_samples)
    nbr_col = (rng.integers(1, k_eff, size=n_samples)
               if k_eff > 1 else np.zeros(n_samples, dtype=int))
    nbr_idx = indices[base_idx, nbr_col]
    base = X_real[base_idx]
    nbr = X_real[nbr_idx]
    alpha = rng.random((n_samples, 1), dtype=np.float32)
    return (base + alpha * (nbr - base)).astype(np.float32)


# Build a CONFIG snapshot that captures everything the generator depends on.
def _vae_bgm_gen_config(label, real_count):
    use_smote = real_count < CONFIG['min_real_for_vae']
    use_anchored = (CONFIG['use_anchored_for_small']
                    and not use_smote
                    and real_count < CONFIG['anchored_threshold'])
    method = ('SMOTE' if use_smote
              else 'VAE-anchored' if use_anchored else 'VAE-BGM')
    cap = class_vae_capacity(real_count)
    return {
        'method': method,
        'capacity': cap,
        'config_subset': {k: CONFIG[k] for k in [
            'latent_dim', 'hidden_size', 'n_epochs', 'batch_size', 'lr',
            'kl_warmup_epochs', 'warmup_lr_epochs', 'grad_clip',
            'weight_decay', 'input_noise', 'n_seeds', 'kl_free_bits',
            'val_split', 'winsorize_pct', 'target_count', 'min_real_for_vae',
            'smote_k_neighbors', 'bgm_n_components_min', 'bgm_n_components_max',
            'bgm_prune_weight', 'decoder_std_scale', 'quantile_match',
            'use_anchored_for_small', 'anchored_threshold',
            'anchored_k_neighbors',
        ]},
        'bgm_n_components': adaptive_bgm_components(real_count),
        'skip_classes': sorted(SKIP_CLASSES),
    }


synthetic_paths = {}
trained_models = {}
training_histories = {}
gen_methods = {}

total_classes = len(classes_to_generate)
for cls_idx, (label, n_gen) in enumerate(classes_to_generate.items()):
    real_count = int(class_counts[label])
    gen_cfg = _vae_bgm_gen_config(label, real_count)
    method = gen_cfg['method']
    gen_methods[label] = method
    cap = gen_cfg['capacity']

    real_class_df = raw_df[raw_df['label'] == label]
    key = cache_key('vae_bgm', label, real_class_df, feature_cols, gen_cfg, n_gen)
    hit = cache_load('vae_bgm', label, key)

    print(f'\n{"=" * 60}')
    print(f'Class {label} ({cls_idx + 1}/{total_classes}) -- '
          f'{real_count:,} real, need {n_gen:,} syn -- method: {method} '
          f'(latent={cap["latent_dim"]}, hidden={cap["hidden_size"]})')
    print(f'  cache key: vae_bgm/{label}_{key}')
    print(f'{"=" * 60}')

    if hit is not None:
        print(f'  CACHE HIT -> {hit}')
        synthetic_paths[label] = hit
        training_histories[label] = None
        continue

    t0 = time.time()
    X_tr, m_tr, X_va, m_va, norm_stats = prepare_class_data(
        raw_df, label, feature_cols,
        max_samples=CONFIG['max_train_samples'],
        val_split=CONFIG['val_split'],
        winsorize_pct=CONFIG['winsorize_pct'],
    )
    print(f'  Data: {X_tr.shape[0]:,} train, {X_va.shape[0]:,} val')

    if method == 'SMOTE':
        X_real_all = np.vstack([X_tr, X_va])
        del X_va, m_va, m_tr
        gc.collect()
        gen_samples = smote_generate(
            X_real_all, n_gen, k=CONFIG['smote_k_neighbors'], random_state=42)
        del X_real_all, X_tr
        training_histories[label] = None
    else:
        bgm_n = adaptive_bgm_components(real_count)
        best_model = None
        best_val_loss = float('inf')
        best_history = None
        for seed in range(CONFIG['n_seeds']):
            print(f'\n  Seed {seed}:')
            torch.manual_seed(seed)
            np.random.seed(seed)
            if DEVICE.type == 'cuda':
                torch.cuda.manual_seed_all(seed)
            model = CropVAE_BGM(
                n_features=n_features,
                latent_dim=cap['latent_dim'],
                hidden_size=cap['hidden_size'],
                free_bits=CONFIG.get('kl_free_bits', 0.0),
            )
            history = model.fit(
                X_tr, m_tr, X_va, m_va,
                n_epochs=CONFIG['n_epochs'],
                batch_size=CONFIG['batch_size'],
                lr=CONFIG['lr'],
                kl_warmup_epochs=CONFIG['kl_warmup_epochs'],
                warmup_lr_epochs=CONFIG['warmup_lr_epochs'],
                grad_clip=CONFIG['grad_clip'],
                weight_decay=CONFIG['weight_decay'],
                input_noise=CONFIG['input_noise'],
                device=DEVICE,
                verbose=True,
            )
            seed_val = history['best_val']
            print(f'    Seed {seed} best val loss: {seed_val:.3f}')
            if seed_val < best_val_loss:
                del best_model
                best_val_loss = seed_val
                best_model = model
                best_history = history
            else:
                del model
            gc.collect()
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
        del X_va, m_va
        gc.collect()
        print(f'\n  Best seed val loss: {best_val_loss:.3f}')
        if method == 'VAE-anchored':
            gen_samples = best_model.generate_anchored(
                X_tr, n_gen,
                k=CONFIG['anchored_k_neighbors'], device=DEVICE,
                decoder_std_scale=CONFIG['decoder_std_scale'],
                random_state=42,
            )
        else:
            print(f'  Fitting BGM (n_components={bgm_n}) ...')
            best_model.fit_bgm(X_tr, n_components=bgm_n, device=DEVICE)
            gen_samples = best_model.generate(
                n_gen, device=DEVICE,
                prune_weight=CONFIG['bgm_prune_weight'],
                decoder_std_scale=CONFIG['decoder_std_scale'],
            )
        del X_tr, m_tr
        gc.collect()
        if cls_idx < 3:
            trained_models[label] = best_model
        else:
            del best_model
            gc.collect()
        training_histories[label] = best_history

    gen_df = denormalize(gen_samples, feature_cols, norm_stats)
    gen_df['label'] = label
    del gen_samples
    if CONFIG.get('quantile_match', False):
        real_only = raw_df[raw_df['label'] == label][feature_cols].dropna()
        gen_df = match_real_marginals(gen_df, real_only, feature_cols)
        gen_df['label'] = label
        del real_only
    syn_path = cache_save('vae_bgm', label, key, gen_df,
                          {'gen_config': gen_cfg, 'n_gen': int(n_gen)})
    synthetic_paths[label] = syn_path
    del gen_df
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    print(f'  Done in {time.time() - t0:.1f}s. Saved -> {syn_path}')

print(f'\n\n{"=" * 60}')
print('VAE-BGM generation complete.')
for tag in ('SMOTE', 'VAE-anchored', 'VAE-BGM'):
    classes = [l for l, m in gen_methods.items() if m == tag]
    if classes:
        print(f'  {tag:15s} {classes}')
print(f'{"=" * 60}')


## 7. Training Loss Curves


In [ ]:
# Plot training curves only for VAE-BGM classes (SMOTE has no history)
vae_histories = {k: v for k, v in training_histories.items() if v is not None}
n_plots = len(vae_histories)
if n_plots == 0:
    print('No VAE-BGM histories to plot (all classes used SMOTE).')
else:
    cols = min(3, n_plots)
    rows = int(np.ceil(n_plots / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
    if n_plots == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for idx, (label, hist) in enumerate(vae_histories.items()):
        ax = axes[idx]
        ax.plot(hist['train_loss'], label='Train', alpha=0.7)
        ax.plot(hist['val_loss'], label='Val', alpha=0.7)
        ax.set_title(f'Class {label}')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.legend()
        ax.set_yscale('log')

    for idx in range(n_plots, len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle('VAE Training Loss Curves (VAE-BGM classes only)', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
    plt.show()

## 8. Validate Synthetic Data Quality


In [ ]:
# 7a. Feature Distribution Comparison (KS test + histograms)

for label in classes_to_generate:
    real_class = raw_df[raw_df['label'] == label][feature_cols].dropna()
    syn_class = pd.read_csv(synthetic_paths[label], usecols=feature_cols)

    n_plot = min(5000, len(real_class), len(syn_class))
    real_sample = real_class.sample(n=n_plot, random_state=42)
    syn_sample = syn_class.sample(n=n_plot, random_state=42)
    del syn_class

    print(f'\nClass {label} — KS Test (p > 0.05 = distributions similar):')
    ks_results = []
    for col in feature_cols:
        stat, pval = stats.ks_2samp(real_sample[col].values, syn_sample[col].values)
        ks_results.append((col, stat, pval))
        status = 'PASS' if pval > 0.05 else 'FAIL'
        print(f'  {col:>12s}: KS={stat:.4f}, p={pval:.4f} [{status}]')

    fig, axes = plt.subplots(3, 5, figsize=(20, 10))
    axes = axes.flatten()
    for idx, col in enumerate(feature_cols):
        ax = axes[idx]
        ax.hist(real_sample[col], bins=50, alpha=0.5, label='Real', density=True, color='steelblue')
        ax.hist(syn_sample[col], bins=50, alpha=0.5, label='Synthetic', density=True, color='coral')
        ax.set_title(col, fontsize=9)
        ax.legend(fontsize=7)
    plt.suptitle(f'Class {label}: Real vs Synthetic Feature Distributions', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'dist_comparison_{label}.png'), dpi=150)
    plt.show()
    plt.close('all')
    gc.collect()


In [ ]:
# 7b. Latent Space Visualization (t-SNE) — VAE-BGM classes only
# Only classes whose model was kept in `trained_models` (cls_idx < 3 in
# main loop, AND not a cache hit). Cache hits don't retrain so the model
# is never instantiated -- skip those rather than load a non-existent
# `model.pt` (we never write one to disk; cache stores synthetic CSV only).

vae_classes = [l for l, m in gen_methods.items() if m == 'VAE-BGM']
plot_classes = [l for l in vae_classes if l in trained_models][:3]

if not vae_classes:
    print('No VAE-BGM classes to plot t-SNE for (all SMOTE).')
elif not plot_classes:
    print('VAE-BGM classes exist but none in memory '
          '(all cache hits or evicted). '
          'Clear cache_v18/vae_bgm/ to retrain and repopulate trained_models.')

for label in plot_classes:
    model = trained_models[label]

    X_tr, m_tr, X_va, m_va, norm_stats = prepare_class_data(
        raw_df, label, feature_cols, max_samples=CONFIG['max_train_samples'])
    del m_tr, X_va, m_va

    n_pts = min(500, X_tr.shape[0])
    model.eval()
    with torch.no_grad():
        x_t = torch.from_numpy(X_tr[:n_pts]).to(DEVICE).float()
        latent_out = model.encoder(x_t)
        lp = model.latent_space.get_latent_params(latent_out)
        z_real = model.latent_space.sample_latent(lp).cpu().numpy()
    del X_tr, x_t, latent_out, lp

    if model.bgm is None:
        bgm_n = adaptive_bgm_components(int(class_counts[label]))
        X_tr2, m_tr2, _, _, _ = prepare_class_data(
            raw_df, label, feature_cols, max_samples=CONFIG['max_train_samples'])
        model.fit_bgm(X_tr2, n_components=bgm_n, device=DEVICE)
        del X_tr2, m_tr2; gc.collect()

    z_bgm = model.bgm.sample(n_pts)[0]
    z_gauss = np.random.randn(n_pts, model.latent_dim)

    z_all = np.vstack([z_real, z_bgm, z_gauss])
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    z_2d = tsne.fit_transform(z_all)
    del z_all, z_bgm, z_gauss, z_real

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(z_2d[:n_pts, 0], z_2d[:n_pts, 1], alpha=0.4, label='Real (VAE z)', s=15)
    ax.scatter(z_2d[n_pts:2*n_pts, 0], z_2d[n_pts:2*n_pts, 1], alpha=0.4, label='BGM (Ours)', s=15)
    ax.scatter(z_2d[2*n_pts:, 0], z_2d[2*n_pts:, 1], alpha=0.4, label='Gaussian (TVAE)', s=15)
    ax.legend()
    ax.set_title(f'Class {label}: Latent Space Comparison (t-SNE)')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'latent_tsne_{label}.png'), dpi=150)
    plt.show()
    plt.close('all')
    del z_2d; gc.collect()

In [ ]:
# 7c. Discriminative Validation (RF real-vs-synthetic classifier)
print('Discriminative Validation (RF accuracy — lower = better, 0.5 = ideal)')
print('=' * 70)

disc_results = {}
for label in classes_to_generate:
    real_class = raw_df[raw_df['label'] == label][feature_cols].dropna().reset_index(drop=True)
    syn_class = pd.read_csv(synthetic_paths[label], usecols=feature_cols).reset_index(drop=True)

    n = min(len(real_class), len(syn_class))
    real_sub = real_class.sample(n=n, random_state=42).reset_index(drop=True)
    syn_sub = syn_class.sample(n=n, random_state=42).reset_index(drop=True)
    del real_class, syn_class

    real_sub['is_synthetic'] = 0
    syn_sub['is_synthetic'] = 1
    combined = pd.concat([real_sub, syn_sub]).sample(frac=1, random_state=42).reset_index(drop=True)
    del real_sub, syn_sub

    X = combined[feature_cols].values
    y = combined['is_synthetic'].values
    del combined

    X_tr2, X_te, y_tr2, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    del X, y

    rf = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
    rf.fit(X_tr2, y_tr2)
    acc = rf.score(X_te, y_te)
    disc_results[label] = acc
    del rf, X_tr2, X_te, y_tr2, y_te
    gc.collect()

    quality = 'EXCELLENT' if acc < 0.6 else ('GOOD' if acc < 0.7 else ('OK' if acc < 0.8 else 'POOR'))
    print(f'  Class {label}: accuracy = {acc:.4f}  [{quality}]')

avg_acc = np.mean(list(disc_results.values()))
print(f'\n  Average discriminative accuracy: {avg_acc:.4f}')


## 9. Correlation Preservation Check


In [ ]:
# Check that inter-feature correlations are preserved in synthetic data
for label in list(classes_to_generate.keys())[:3]:
    real_class = raw_df[raw_df['label'] == label][feature_cols].dropna()
    syn_class = pd.read_csv(synthetic_paths[label], usecols=feature_cols)

    corr_real = real_class.corr()
    corr_syn = syn_class.corr()
    del real_class, syn_class

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(20, 5))

    sns.heatmap(corr_real, ax=ax1, cmap='coolwarm', vmin=-1, vmax=1,
                xticklabels=True, yticklabels=True, square=True)
    ax1.set_title(f'Real (Class {label})')
    ax1.tick_params(labelsize=6)

    sns.heatmap(corr_syn, ax=ax2, cmap='coolwarm', vmin=-1, vmax=1,
                xticklabels=True, yticklabels=True, square=True)
    ax2.set_title(f'Synthetic (Class {label})')
    ax2.tick_params(labelsize=6)

    diff = (corr_real - corr_syn).abs()
    sns.heatmap(diff, ax=ax3, cmap='Reds', vmin=0, vmax=0.5,
                xticklabels=True, yticklabels=True, square=True)
    ax3.set_title(f'|Difference|  (mean={diff.mean().mean():.3f})')
    ax3.tick_params(labelsize=6)

    plt.suptitle(f'Correlation Matrices — Class {label}', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'correlation_{label}.png'), dpi=150)
    plt.show()
    plt.close('all')
    del corr_real, corr_syn, diff; gc.collect()


## 10. RF Baseline + Augmented (paper-exact)


In [ ]:
# =====================================================================
# Section 9 — REFACTORED v7: PAPER-EXACT baseline
#   Reference: Kajornchaikitti et al. 2026, "Crop Classification Using
#   Random Forest and Sentinel-2 Imagery" — see randomForest_model.ipynb.
#
# v7 fix vs result7.pdf (baseline 0.60, paper Table IV reports 0.7157 acc):
#   - Test set was BALANCED 5K/class (TEST_PER_CLASS=5000) -> rare classes
#     dragged weighted F1 down to 0.60. Paper uses STRATIFIED 80:10:10 with
#     NATURAL class distribution -> big well-classified classes (F1>0.7)
#     dominate the weighted average -> 0.71.
#   - Drop max_samples=0.7 (paper uses default full bootstrap).
#   - Add validation metrics (paper reports both val+test in Table IV).
#   - Add Kappa + Precision + Recall to match paper's metric set exactly.
#
# Paper code (reproduced verbatim from randomForest_model.ipynb):
#     df_clean = df.dropna()
#     X_train, X_temp = train_test_split(test_size=0.2, stratify=y, random_state=42)
#     X_val,   X_test = train_test_split(temp,  test_size=0.5, stratify=y_temp, random_state=42)
#     RF(n_estimators=254, max_depth=36, min_samples_split=5, min_samples_leaf=1,
#        max_features='log2', class_weight='balanced', random_state=42, n_jobs=-1)
# =====================================================================
from sklearn.metrics import precision_score, recall_score, cohen_kappa_score

del trained_models
gc.collect()

# RF hyper-params — PAPER EXACT (Table III)
RF_KW = dict(
    n_estimators=254,
    max_depth=36,
    min_samples_leaf=1,
    min_samples_split=5,
    max_features='log2',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,                # set to 2 + add max_samples=0.7 if RAM-constrained
)
print(f'RF settings (paper exact): {RF_KW}')

# Drop NaN, downcast, stratified 80/10/10 split
df_clean = raw_df.dropna(subset=feature_cols).reset_index(drop=True)
n_dropped = len(raw_df) - len(df_clean)
print(f'NaN rows dropped: {n_dropped:,} ({100 * n_dropped / len(raw_df):.1f}%)')
print(f'Clean dataset:    {len(df_clean):,} samples')

for c in feature_cols:
    df_clean[c] = df_clean[c].astype(np.float32)

train_df, temp_df = train_test_split(
    df_clean, test_size=0.2, random_state=42, stratify=df_clean['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['label']
)
del df_clean, temp_df
gc.collect()

X_train_real = train_df[feature_cols].to_numpy(dtype=np.float32, copy=False)
y_train_real = train_df['label'].to_numpy()
X_val = val_df[feature_cols].to_numpy(dtype=np.float32, copy=False)
y_val = val_df['label'].to_numpy()
X_test = test_df[feature_cols].to_numpy(dtype=np.float32, copy=False)
y_test = test_df['label'].to_numpy()
del val_df, test_df
gc.collect()

real_counts = pd.Series(y_train_real).value_counts()
print(f'\nTrain: {len(X_train_real):,}   Val: {len(X_val):,}   Test: {len(X_test):,}')
print(f'Per-class real-train counts (paper 80% split):')
for lbl, cnt in real_counts.sort_index().items():
    print(f'    {lbl}: {cnt:>9,}')

# --- Baseline RF (real only) ---
print(f'\nFitting baseline RF (paper exact, 80:10:10, NaN-dropped)...')
t0 = time.time()
rf_real = RandomForestClassifier(**RF_KW)
rf_real.fit(X_train_real, y_train_real)
print(f'Baseline trained in {time.time() - t0:.1f}s')

y_val_pred_real = rf_real.predict(X_val)
y_pred_real = rf_real.predict(X_test)
del rf_real, X_train_real, y_train_real
gc.collect()


def _print_paper_metrics(y_true, y_pred, header, paper_ref=None):
    """Print metrics in paper's order: Acc, Kappa, F1, Precision, Recall.
    paper_ref: optional dict of paper values to display alongside."""
    print(f'\n>>> {header} <<<')
    metrics = {
        'Accuracy':     accuracy_score(y_true, y_pred),
        'Kappa':        cohen_kappa_score(y_true, y_pred),
        'Weighted F1':  f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'Precision':    precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'Recall':       recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'Macro F1':     f1_score(y_true, y_pred, average='macro', zero_division=0),
        'Balanced Acc': balanced_accuracy_score(y_true, y_pred),
    }
    for k, v in metrics.items():
        suffix = f'    [paper: {paper_ref[k]:.4f}]' if paper_ref and k in paper_ref else ''
        print(f'{k:<14s} {v:.4f}{suffix}')
    return metrics


print('\n--- BASELINE: Real Data Only (paper-exact RF) ---')
_print_paper_metrics(y_val, y_val_pred_real, 'Validation set (paper Table IV)',
                     paper_ref={'Accuracy': 0.7174, 'Kappa': 0.6795,
                                'Weighted F1': 0.7155, 'Precision': 0.7273,
                                'Recall': 0.7174})
print('\nTest set classification report:')
print(classification_report(y_test, y_pred_real, zero_division=0))
_print_paper_metrics(y_test, y_pred_real, 'Test set (paper Table IV)',
                     paper_ref={'Accuracy': 0.7157, 'Kappa': 0.6775,
                                'Weighted F1': 0.7139, 'Precision': 0.7260,
                                'Recall': 0.7157})

MAX_SYN_RATIO = CONFIG['max_syn_ratio']


def _build_aug_rf_eval(synthetic_paths_dict, method_name):
    """Augment train_df with capped synthetic from a paths dict, fit paper-spec
    RF, predict on the same X_test from the baseline cell. Returns y_pred."""
    aug_chunks = [train_df]
    syn_added = {}
    for label, syn_path in synthetic_paths_dict.items():
        real_count = int(real_counts.get(label, 0))
        needed = MAX_SYN_RATIO * real_count
        if needed <= 0:
            continue
        syn = pd.read_csv(syn_path)
        if len(syn) > needed:
            syn = syn.sample(n=needed, random_state=42)
        for c in feature_cols:
            syn[c] = syn[c].astype(np.float32)
        aug_chunks.append(syn[feature_cols + ['label']])
        syn_added[label] = len(syn)

    aug_df = pd.concat(aug_chunks, ignore_index=True)
    del aug_chunks
    aug_df = aug_df.sample(frac=1, random_state=42).reset_index(drop=True)
    Xa = aug_df[feature_cols].to_numpy(dtype=np.float32, copy=False)
    ya = aug_df['label'].to_numpy()
    del aug_df
    gc.collect()

    print(f'\n[{method_name}] Train: {len(Xa):,} samples '
          f'({Xa.nbytes / 1e6:.1f} MB)')
    print(f'  Synthetic added per class (cap = {MAX_SYN_RATIO}x real):')
    for lbl, n in sorted(syn_added.items()):
        rn = int(real_counts.get(lbl, 0))
        gen_method = gen_methods.get(lbl, method_name) if method_name == 'VAE-BGM' else method_name
        print(f'    {lbl}: {rn:>7,} real + {n:>7,} {gen_method:<8s} '
              f'= {rn+n:>7,} ({n/max(1,rn):.1f}x)')

    print(f'  Fitting {method_name}-augmented RF (paper exact)...')
    t0 = time.time()
    rf = RandomForestClassifier(**RF_KW)
    rf.fit(Xa, ya)
    print(f'  trained in {time.time() - t0:.1f}s')
    yp = rf.predict(X_test)
    del rf, Xa, ya
    gc.collect()
    return yp

In [ ]:
# Augmented: real + capped synthetic (VAE-BGM/SMOTE)
y_pred_aug = _build_aug_rf_eval(synthetic_paths, 'VAE-BGM')
print('\n--- AUGMENTED: Real + (SMOTE/VAE-BGM) Synthetic ---')
print(classification_report(y_test, y_pred_aug, zero_division=0))
_print_paper_metrics(y_test, y_pred_aug, 'Test set — Real + VAE-BGM/SMOTE')

In [ ]:
# Side-by-side comparison
labels_sorted = sorted(class_counts.keys())
f1_real = f1_score(y_test, y_pred_real, labels=labels_sorted, average=None, zero_division=0)
f1_aug = f1_score(y_test, y_pred_aug, labels=labels_sorted, average=None, zero_division=0)

comparison = pd.DataFrame({
    'Class': labels_sorted,
    'Count': [class_counts[l] for l in labels_sorted],
    'F1_Real': np.round(f1_real, 4),
    'F1_Augmented': np.round(f1_aug, 4),
    'F1_Diff': np.round(f1_aug - f1_real, 4),
    'Generated': ['Yes' if l in classes_to_generate else 'No' for l in labels_sorted]
})

print('Per-class F1 comparison:')
print(comparison.to_string(index=False))

print(f'\nMacro F1 (real only):      {f1_score(y_test, y_pred_real, average="macro", zero_division=0):.4f}')
print(f'Macro F1 (augmented):      {f1_score(y_test, y_pred_aug, average="macro", zero_division=0):.4f}')
print(f'Balanced Acc (real only):  {balanced_accuracy_score(y_test, y_pred_real):.4f}')
print(f'Balanced Acc (augmented):  {balanced_accuracy_score(y_test, y_pred_aug):.4f}')

# Plot F1 comparison
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(labels_sorted))
w = 0.35
ax.bar(x - w/2, f1_real, w, label='Real Only', color='steelblue')
ax.bar(x + w/2, f1_aug, w, label='Real + VAE-BGM', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(labels_sorted, rotation=45)
ax.set_xlabel('Crop Class')
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1: Real Only vs Real + VAE-BGM Synthetic')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'f1_comparison.png'), dpi=150)
plt.show()

## 11. Export Balanced Dataset


In [ ]:
# Save balanced dataset — load synthetic per-class from disk, stream to avoid big concat in RAM
target = CONFIG['target_count']
balanced_path = os.path.join(OUTPUT_DIR, 'balanced_dataset.csv')

first = True
for label in class_counts.index:  # v10 BUGFIX: was iterating Series VALUES (counts), not labels -> balanced_dataset.csv was empty (result11 Shape (0,1))
    real_class = raw_df[raw_df['label'] == label]
    if label in synthetic_paths:
        syn_class = pd.read_csv(synthetic_paths[label])
        combined = pd.concat([real_class, syn_class], ignore_index=True)
        del syn_class
    else:
        combined = real_class.copy()
    if len(combined) > target:
        combined = combined.sample(n=target, random_state=42)
    combined.to_csv(balanced_path, mode='w' if first else 'a', header=first, index=False)
    first = False
    del combined; gc.collect()

# Save all_synthetic.csv
synthetic_chunks = [pd.read_csv(p) for p in synthetic_paths.values()]
synthetic_df = pd.concat(synthetic_chunks, ignore_index=True)
del synthetic_chunks
synthetic_path = os.path.join(OUTPUT_DIR, 'all_synthetic.csv')
synthetic_df.to_csv(synthetic_path, index=False)

print(f'Synthetic data saved: {synthetic_path}')
print(f'  Shape: {synthetic_df.shape}')
print(f'  Classes: {synthetic_df["label"].value_counts().to_dict()}')
del synthetic_df; gc.collect()

print(f'\nBalanced dataset saved: {balanced_path}')
bal_check = pd.read_csv(balanced_path, usecols=['label'])
print(f'  Shape: {bal_check.shape}')
print(f'  Class distribution:')
for lbl, cnt in bal_check['label'].value_counts().sort_index().items():
    print(f'    {lbl}: {cnt:>10,}')
del bal_check


## 12. Confusion Matrices


In [ ]:
# Confusion matrices side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

cm_real = confusion_matrix(y_test, y_pred_real, labels=labels_sorted)
cm_aug = confusion_matrix(y_test, y_pred_aug, labels=labels_sorted)

ConfusionMatrixDisplay(cm_real, display_labels=labels_sorted).plot(ax=ax1, cmap='Blues', values_format='d')
ax1.set_title('Real Only (class_weight=balanced)')
ax1.tick_params(labelrotation=45)

ConfusionMatrixDisplay(cm_aug, display_labels=labels_sorted).plot(ax=ax2, cmap='Oranges', values_format='d')
ax2.set_title('Real + VAE-BGM Synthetic')
ax2.tick_params(labelrotation=45)

plt.suptitle('Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrices.png'), dpi=150)
plt.show()

## 13. TVAE Baseline (Xu NeurIPS 2019)


In [ ]:
# =====================================================================
# TVAE BASELINE (v19 -- cached + SMOTE fallback for tiny classes)
#   TVAE = Tabular VAE (Xu et al. NeurIPS 2019). Same paper as CTGAN, but
#   the VAE variant. Used here as apples-to-apples vs our VAE-BGM.
#
#   v18: same content-hash cache as VAE-BGM. Re-running this cell with
#   no config change is a no-op.
#
#   v19 (2026-05-03): SMOTE fallback when real_count < min_real_for_vae.
#     result18_analysis.md showed VAE-BGM's class-2420 win was actually
#     SMOTE doing the work (VAE-BGM pipeline auto-routes to SMOTE for
#     N<5K). TVAE/TabDDPM/TabSyn lacked the same fallback, so they
#     trained neural nets on 1.6K samples -> off-manifold synth (disc
#     RF 0.85+) -> no F1 gain. v19 wires the same SMOTE path into all
#     three baseline pipelines so the next 5-way comparison measures
#     generator quality at N>=5K, not pipeline-routing differences.
#     gen_cfg now includes 'fallback' field, so v19 cache keys differ
#     from v18 keys for tiny classes only -- big-class caches still hit.
# =====================================================================
try:
    from ctgan import TVAE
    HAS_TVAE = True
except ImportError:
    print('TVAE lives in `ctgan` package. Run: pip install ctgan')
    HAS_TVAE = False

TVAE_CONFIG = {
    'epochs': 150,
    'batch_size': 500,
    'embedding_dim': 128,
    'compress_dims': (128, 128),
    'decompress_dims': (128, 128),
    'max_train': 30000,
    'winsorize_pct': CONFIG['winsorize_pct'],
}

tvae_paths = {}

if HAS_TVAE:
    cuda = (DEVICE.type == 'cuda')
    for cls_idx, (label, n_gen) in enumerate(classes_to_generate.items()):
        real_count = int(class_counts[label])
        use_smote_fb = real_count < CONFIG['min_real_for_vae']
        gen_cfg = {'method': 'TVAE', 'config': TVAE_CONFIG,
                   'fallback': 'SMOTE' if use_smote_fb else None}
        real_class_df = raw_df[raw_df['label'] == label]
        key = cache_key('tvae', label, real_class_df, feature_cols, gen_cfg, n_gen)
        hit = cache_load('tvae', label, key)
        method_str = 'SMOTE-fb' if use_smote_fb else 'TVAE'
        print(f'[TVAE] Class {label} ({cls_idx + 1}/{len(classes_to_generate)}) -- '
              f'{real_count:,} real, gen {n_gen:,}  '
              f'method={method_str}  cache: tvae/{label}_{key}')
        if hit is not None:
            print(f'  CACHE HIT -> {hit}')
            tvae_paths[label] = hit
            continue

        # v19: SMOTE fallback for tiny classes -- no neural training.
        if use_smote_fb:
            t0 = time.time()
            cls_real = (raw_df[raw_df['label'] == label][feature_cols]
                        .dropna().to_numpy(dtype=np.float32))
            syn_arr = smote_generate(
                cls_real, n_gen,
                k=CONFIG['smote_k_neighbors'], random_state=42)
            syn_df = pd.DataFrame(syn_arr, columns=feature_cols)
            syn_df['label'] = label
            syn_path = cache_save(
                'tvae', label, key, syn_df,
                {'gen_config': gen_cfg, 'n_gen': int(n_gen),
                 'fallback': 'SMOTE', 'real_count': real_count})
            tvae_paths[label] = syn_path
            print(f'  v19 SMOTE fallback (N={real_count:,} '
                  f'< min_real_for_vae={CONFIG["min_real_for_vae"]:,}) '
                  f'-> {len(syn_df):,} samples in '
                  f'{time.time() - t0:.1f}s -> {syn_path}')
            del cls_real, syn_arr, syn_df
            gc.collect()
            continue

        t0 = time.time()
        cls_df = (raw_df[raw_df['label'] == label][feature_cols]
                  .dropna().reset_index(drop=True))
        wp = TVAE_CONFIG['winsorize_pct']
        if wp > 0:
            for col in feature_cols:
                lo = cls_df[col].quantile(wp)
                hi = cls_df[col].quantile(1.0 - wp)
                cls_df[col] = cls_df[col].clip(lower=lo, upper=hi)
        if len(cls_df) > TVAE_CONFIG['max_train']:
            cls_df = cls_df.sample(n=TVAE_CONFIG['max_train'],
                                   random_state=42).reset_index(drop=True)

        bs = min(TVAE_CONFIG['batch_size'], max(64, len(cls_df) // 4))
        epochs = TVAE_CONFIG['epochs']

        model = TVAE(
            epochs=epochs,
            batch_size=bs,
            embedding_dim=TVAE_CONFIG['embedding_dim'],
            compress_dims=TVAE_CONFIG['compress_dims'],
            decompress_dims=TVAE_CONFIG['decompress_dims'],
            cuda=cuda,
        )
        model.fit(cls_df, discrete_columns=[])
        syn = model.sample(n_gen)
        syn['label'] = label
        syn_path = cache_save('tvae', label, key, syn,
                              {'gen_config': gen_cfg, 'n_gen': int(n_gen),
                               'epochs': epochs, 'batch_size': bs,
                               'real_count': real_count})
        tvae_paths[label] = syn_path
        print(f'  trained ep={epochs} bs={bs} train={len(cls_df):,} -> '
              f'{len(syn):,} in {time.time() - t0:.1f}s -> {syn_path}')
        del model, syn, cls_df
        gc.collect()
        if cuda:
            torch.cuda.empty_cache()

    print(f'\nTVAE done. {len(tvae_paths)} classes.')
else:
    print('Skipping TVAE (package missing).')


In [ ]:
# 13a. TVAE quality validation — KS pass rate + discriminative RF
if HAS_TVAE and tvae_paths:
    print('TVAE — KS pass rate + discriminative RF (lower = better, 0.5 = ideal)')
    print('=' * 70)

    tvae_disc = {}
    tvae_ks_pass = {}
    for label in classes_to_generate:
        if label not in tvae_paths:
            continue
        real_class = (raw_df[raw_df['label'] == label][feature_cols]
                      .dropna().reset_index(drop=True))
        syn_class = pd.read_csv(tvae_paths[label], usecols=feature_cols).reset_index(drop=True)

        n_ks = min(5000, len(real_class), len(syn_class))
        rs = real_class.sample(n=n_ks, random_state=42)
        ss = syn_class.sample(n=n_ks, random_state=42)
        n_pass = sum(
            1 for col in feature_cols
            if (lambda r: r[1] > 0.05 or r[0] < 0.10)(stats.ks_2samp(rs[col].values, ss[col].values))
        )
        ks_rate = n_pass / len(feature_cols)
        tvae_ks_pass[label] = ks_rate

        n = min(len(real_class), len(syn_class))
        real_sub = real_class.sample(n=n, random_state=42).reset_index(drop=True)
        syn_sub = syn_class.sample(n=n, random_state=42).reset_index(drop=True)
        del real_class, syn_class
        real_sub['is_synthetic'] = 0
        syn_sub['is_synthetic'] = 1
        comb = pd.concat([real_sub, syn_sub]).sample(frac=1, random_state=42).reset_index(drop=True)
        del real_sub, syn_sub
        Xd = comb[feature_cols].values
        yd = comb['is_synthetic'].values
        del comb
        Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(
            Xd, yd, test_size=0.2, random_state=42, stratify=yd)
        del Xd, yd
        rf_d = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
        rf_d.fit(Xd_tr, yd_tr)
        acc = rf_d.score(Xd_te, yd_te)
        tvae_disc[label] = acc
        del rf_d, Xd_tr, Xd_te, yd_tr, yd_te
        gc.collect()
        q = ('EXCELLENT' if acc < 0.6 else
             ('GOOD' if acc < 0.7 else
              ('OK' if acc < 0.8 else 'POOR')))
        print(f'  Class {label}: KS pass = {ks_rate*100:5.1f}%   '
              f'disc RF = {acc:.4f}  [{q}]')

    print(f'\n  Avg KS pass rate: {np.mean(list(tvae_ks_pass.values()))*100:.1f}%')
    print(f'  Avg disc RF acc:  {np.mean(list(tvae_disc.values())):.4f}')
else:
    print('Skipping TVAE quality eval (no tvae_paths).')

In [ ]:
# Augmented RF -- Real + TVAE synthetic
if HAS_TVAE and tvae_paths:
    y_pred_t = _build_aug_rf_eval(tvae_paths, 'TVAE')
    print('\n--- AUGMENTED: Real + TVAE Synthetic ---')
    print(classification_report(y_test, y_pred_t, zero_division=0))
    _print_paper_metrics(y_test, y_pred_t, 'Test set -- Real + TVAE')
else:
    print('Skipping TVAE augmented RF (no tvae_paths).')


## 14. TabDDPM Model Definition (Kotelnikov ICML 2023)


In [ ]:
# =====================================================================
# TabDDPM -- Tabular Denoising Diffusion Probabilistic Model
#   Kotelnikov et al. ICML 2023 + Ho et al. 2020 + Nichol&Dhariwal 2021
#
#   Components:
#     - cosine_beta_schedule    : Nichol&Dhariwal 2021 cosine SNR schedule
#     - SinusoidalTimeEmbedding : Vaswani 2017 positional enc for timestep
#     - TabDDPMDenoiser         : MLP epsilon-predictor with residual blocks
#     - TabDDPM                 : full diffusion model (forward + sample + fit)
# =====================================================================

import math


def cosine_beta_schedule(T, s=0.008):
    """Cosine schedule from Nichol & Dhariwal 2021 (Improved DDPM Eq. 17)."""
    steps = T + 1
    t = torch.linspace(0, T, steps) / T
    alpha_bar = torch.cos((t + s) / (1 + s) * math.pi * 0.5) ** 2
    alpha_bar = alpha_bar / alpha_bar[0]
    betas = 1 - (alpha_bar[1:] / alpha_bar[:-1])
    return torch.clamp(betas, 1e-5, 0.999)


class SinusoidalTimeEmbedding(nn.Module):
    """Sinusoidal positional encoding for diffusion timestep (Vaswani 2017)."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, device=t.device).float() / half)
        args = t.float()[:, None] * freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class TabDDPMDenoiser(nn.Module):
    """MLP epsilon-predictor: (x_t, t) -> noise.

    Residual blocks with LayerNorm + SiLU + FiLM-style timestep injection.
    Kotelnikov 2023 found this beats simple concat-time MLPs.
    """
    def __init__(self, n_features, hidden=256, time_emb_dim=128, n_blocks=4, dropout=0.1):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmbedding(time_emb_dim // 2),
            nn.Linear(time_emb_dim // 2, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )
        self.in_proj = nn.Linear(n_features, hidden)
        self.blocks = nn.ModuleList([
            nn.ModuleDict({
                'norm': nn.LayerNorm(hidden),
                'fc1': nn.Linear(hidden, hidden),
                'fc2': nn.Linear(hidden, hidden),
                't_proj': nn.Linear(time_emb_dim, hidden),
                'drop': nn.Dropout(dropout),
            })
            for _ in range(n_blocks)
        ])
        self.out_norm = nn.LayerNorm(hidden)
        self.out_proj = nn.Linear(hidden, n_features)

    def forward(self, x, t):
        h = self.in_proj(x)
        t_emb = self.time_mlp(t)
        for blk in self.blocks:
            res = h
            h = blk['norm'](h)
            h = F.silu(blk['fc1'](h)) + blk['t_proj'](t_emb)
            h = blk['drop'](h)
            h = blk['fc2'](h)
            h = h + res
        h = self.out_norm(h)
        return self.out_proj(h)


class TabDDPM(nn.Module):
    """Gaussian diffusion model for continuous tabular data.

    Forward q(x_t | x_0) = N(sqrt(a_bar_t) x_0, (1 - a_bar_t) I)
    Reverse p_theta(x_{t-1} | x_t) parameterised via epsilon-prediction.
    Loss: simplified MSE on noise (Ho 2020 Eq. 14).
    """
    def __init__(self, n_features, T=1000, hidden=256, time_emb_dim=128,
                 n_blocks=4, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.T = T
        self.denoiser = TabDDPMDenoiser(
            n_features, hidden=hidden, time_emb_dim=time_emb_dim,
            n_blocks=n_blocks, dropout=dropout,
        )
        betas = cosine_beta_schedule(T)
        alphas = 1.0 - betas
        alpha_bars = torch.cumprod(alphas, dim=0)
        self.register_buffer('betas', betas)
        self.register_buffer('alphas', alphas)
        self.register_buffer('alpha_bars', alpha_bars)
        self.register_buffer('sqrt_alpha_bars', torch.sqrt(alpha_bars))
        self.register_buffer('sqrt_one_minus_alpha_bars', torch.sqrt(1.0 - alpha_bars))

    def q_sample(self, x0, t, noise):
        sa = self.sqrt_alpha_bars[t].unsqueeze(-1)
        som = self.sqrt_one_minus_alpha_bars[t].unsqueeze(-1)
        return sa * x0 + som * noise

    def loss(self, x0):
        b = x0.shape[0]
        t = torch.randint(0, self.T, (b,), device=x0.device)
        noise = torch.randn_like(x0)
        x_t = self.q_sample(x0, t, noise)
        pred = self.denoiser(x_t, t)
        return F.mse_loss(pred, noise)

    @torch.no_grad()
    def sample(self, n_samples, device=None, batch=4096):
        """DDPM ancestral sampling (Ho 2020 Algorithm 2)."""
        if device is None:
            device = next(self.parameters()).device
        self.eval()
        out = []
        remaining = n_samples
        while remaining > 0:
            b = min(batch, remaining)
            x = torch.randn(b, self.n_features, device=device)
            for t in reversed(range(self.T)):
                t_b = torch.full((b,), t, device=device, dtype=torch.long)
                pred_noise = self.denoiser(x, t_b)
                alpha_t = self.alphas[t]
                alpha_bar_t = self.alpha_bars[t]
                beta_t = self.betas[t]
                coef = beta_t / torch.sqrt(1.0 - alpha_bar_t)
                mean = (1.0 / torch.sqrt(alpha_t)) * (x - coef * pred_noise)
                if t > 0:
                    sigma = torch.sqrt(beta_t)
                    x = mean + sigma * torch.randn_like(x)
                else:
                    x = mean
            out.append(x.cpu().numpy())
            remaining -= b
        return np.vstack(out)

    def fit(self, X_train, X_val, n_epochs=400, batch_size=4096, lr=1e-3,
            weight_decay=1e-4, grad_clip=5.0, patience=40, min_delta=1e-4,
            device=DEVICE, verbose=True):
        """Train with Adam + ReduceLROnPlateau + early stop on val MSE."""
        self.to(device)
        opt = torch.optim.Adam(self.parameters(), lr=lr, weight_decay=weight_decay)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode='min', factor=0.5, patience=15, min_lr=1e-5,
        )
        Xt = torch.as_tensor(X_train, dtype=torch.float32, device=device)
        Xv = torch.as_tensor(X_val, dtype=torch.float32, device=device)
        n = Xt.shape[0]
        best_val = float('inf')
        best_state = None
        wait = 0
        history = {'train': [], 'val': []}
        for epoch in range(n_epochs):
            self.train()
            perm = torch.randperm(n, device=device)
            tot = 0.0
            nb = (n + batch_size - 1) // batch_size
            for i in range(nb):
                idx = perm[i * batch_size:(i + 1) * batch_size]
                if idx.shape[0] < 2:
                    continue
                xb = Xt[idx]
                loss = self.loss(xb)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=grad_clip)
                opt.step()
                tot += loss.item()
            tot /= max(1, nb)
            self.eval()
            with torch.no_grad():
                vl = self.loss(Xv).item()
            history['train'].append(tot)
            history['val'].append(vl)
            sched.step(vl)
            if vl < best_val - min_delta:
                best_val = vl
                best_state = {k: v.detach().cpu().clone() for k, v in self.state_dict().items()}
                wait = 0
            else:
                wait += 1
            if verbose and epoch % 25 == 0:
                lr_now = opt.param_groups[0]['lr']
                print(f'  Epoch {epoch:4d} | train={tot:.4f} | val={vl:.4f} | lr={lr_now:.2e}')
            if wait >= patience:
                if verbose:
                    print(f'  Early stop at epoch {epoch} (best val={best_val:.4f})')
                break
        if best_state is not None:
            self.load_state_dict(best_state)
        del Xt, Xv
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        history['best_val'] = best_val
        return history


print('TabDDPM defined (cosine schedule, T=1000, sinusoidal time emb, MLP residual denoiser).')


## 15. TabDDPM Per-Class Generation + RF Eval


In [ ]:
# =====================================================================
# TabDDPM PER-CLASS GENERATION (v19 -- cached + SMOTE fallback)
# Same model as v17 (Kotelnikov ICML 2023). Now wrapped in disk cache and
# scoped to classes_to_generate (drops v17's 2405/2413 extras -- those
# were for the LightGBM track, removed in v18).
#
# v19 (2026-05-03): SMOTE fallback when real_count < min_real_for_vae.
#   See result18_analysis.md sec 5.1. TabDDPM trained on 1.6K samples
#   produced off-manifold synth that hurt 2420 F1 (-0.007 vs real-only).
#   Now tiny classes route to SMOTE so DDPM is only evaluated where it
#   has enough data to learn the distribution. Big-class behaviour
#   unchanged; only tiny-class cache keys differ from v18.
# =====================================================================
DDPM_CONFIG = {
    'T': 1000,
    'time_emb_dim': 128,
    'dropout': 0.1,
    'n_epochs': 400,
    'batch_size': 4096,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'grad_clip': 5.0,
    'patience': 40,
    'sample_batch': 4096,
    'winsorize_pct': CONFIG['winsorize_pct'],
    'val_split': CONFIG['val_split'],
    'quantile_match': CONFIG['quantile_match'],
}


def ddpm_capacity(n_real):
    if n_real < 5000:
        return {'hidden': 128, 'n_blocks': 3}
    if n_real < 15000:
        return {'hidden': 192, 'n_blocks': 4}
    return {'hidden': 256, 'n_blocks': 4}


synthetic_paths_ddpm = {}
ddpm_histories = {}

for cls_idx, (label, n_gen) in enumerate(classes_to_generate.items()):
    real_count = int(class_counts[label])
    use_smote_fb = real_count < CONFIG['min_real_for_vae']
    cap = ddpm_capacity(real_count)
    gen_cfg = {'method': 'TabDDPM', 'config': DDPM_CONFIG, 'capacity': cap,
               'fallback': 'SMOTE' if use_smote_fb else None}
    real_class_df = raw_df[raw_df['label'] == label]
    key = cache_key('tabddpm', label, real_class_df, feature_cols, gen_cfg, n_gen)
    hit = cache_load('tabddpm', label, key)

    method_str = 'SMOTE-fb' if use_smote_fb else 'TabDDPM'
    print(f'\n[TabDDPM] Class {label} ({cls_idx + 1}/{len(classes_to_generate)}) '
          f'-- {real_count:,} real, gen {n_gen:,}  '
          f'method={method_str}  '
          f'cap=(h={cap["hidden"]}, b={cap["n_blocks"]})  '
          f'cache: tabddpm/{label}_{key}')
    if hit is not None:
        print(f'  CACHE HIT -> {hit}')
        synthetic_paths_ddpm[label] = hit
        continue

    # v19: SMOTE fallback for tiny classes -- no DDPM training.
    if use_smote_fb:
        t0 = time.time()
        cls_real = (raw_df[raw_df['label'] == label][feature_cols]
                    .dropna().to_numpy(dtype=np.float32))
        syn_arr = smote_generate(
            cls_real, n_gen,
            k=CONFIG['smote_k_neighbors'], random_state=42)
        gen_df = pd.DataFrame(syn_arr, columns=feature_cols)
        gen_df['label'] = label
        syn_path = cache_save(
            'tabddpm', label, key, gen_df,
            {'gen_config': gen_cfg, 'n_gen': int(n_gen),
             'fallback': 'SMOTE', 'real_count': real_count})
        synthetic_paths_ddpm[label] = syn_path
        ddpm_histories[label] = None
        print(f'  v19 SMOTE fallback (N={real_count:,} '
              f'< min_real_for_vae={CONFIG["min_real_for_vae"]:,}) '
              f'-> {len(gen_df):,} samples in '
              f'{time.time() - t0:.1f}s -> {syn_path}')
        del cls_real, syn_arr, gen_df
        gc.collect()
        continue

    t0 = time.time()
    X_tr, _, X_va, _, norm_stats = prepare_class_data(
        raw_df, label, feature_cols,
        max_samples=CONFIG['max_train_samples'],
        val_split=DDPM_CONFIG['val_split'],
        winsorize_pct=DDPM_CONFIG['winsorize_pct'],
    )
    print(f'  Data: {X_tr.shape[0]:,} train, {X_va.shape[0]:,} val')

    torch.manual_seed(42)
    np.random.seed(42)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(42)

    model = TabDDPM(
        n_features=n_features,
        T=DDPM_CONFIG['T'],
        hidden=cap['hidden'],
        time_emb_dim=DDPM_CONFIG['time_emb_dim'],
        n_blocks=cap['n_blocks'],
        dropout=DDPM_CONFIG['dropout'],
    )
    history = model.fit(
        X_tr, X_va,
        n_epochs=DDPM_CONFIG['n_epochs'],
        batch_size=DDPM_CONFIG['batch_size'],
        lr=DDPM_CONFIG['lr'],
        weight_decay=DDPM_CONFIG['weight_decay'],
        grad_clip=DDPM_CONFIG['grad_clip'],
        patience=DDPM_CONFIG['patience'],
        device=DEVICE,
        verbose=True,
    )
    ddpm_histories[label] = history
    print(f'  Trained in {time.time() - t0:.1f}s. Best val MSE={history["best_val"]:.4f}')

    print(f'  Sampling {n_gen:,} (T={DDPM_CONFIG["T"]} reverse steps)...')
    t1 = time.time()
    gen_samples = model.sample(n_gen, device=DEVICE, batch=DDPM_CONFIG['sample_batch'])
    print(f'  Sampled in {time.time() - t1:.1f}s')

    gen_df = denormalize(gen_samples, feature_cols, norm_stats)
    gen_df['label'] = label
    if DDPM_CONFIG['quantile_match']:
        real_only = raw_df[raw_df['label'] == label][feature_cols].dropna()
        gen_df = match_real_marginals(gen_df, real_only, feature_cols)
        gen_df['label'] = label
        del real_only

    syn_path = cache_save('tabddpm', label, key, gen_df,
                          {'gen_config': gen_cfg, 'n_gen': int(n_gen),
                           'best_val': history['best_val']})
    synthetic_paths_ddpm[label] = syn_path
    print(f'  Saved -> {syn_path}')
    del model, X_tr, X_va, gen_samples, gen_df
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

print(f'\nTabDDPM done. {len(synthetic_paths_ddpm)} classes.')


In [ ]:
# Augmented RF -- Real + TabDDPM synthetic
if synthetic_paths_ddpm:
    y_pred_d = _build_aug_rf_eval(synthetic_paths_ddpm, 'TabDDPM')
    print('\n--- AUGMENTED: Real + TabDDPM Synthetic ---')
    print(classification_report(y_test, y_pred_d, zero_division=0))
    _print_paper_metrics(y_test, y_pred_d, 'Test set -- Real + TabDDPM')
else:
    print('Skipping TabDDPM augmented RF (no synthetic_paths_ddpm).')


## 16. TabSyn Model Definition (Zhang ICLR 2024 -- Latent Diffusion)


In [ ]:
# =====================================================================
# TabSyn -- Mixed-Type Tabular Synthesis with Score-based Diffusion in
# Latent Space (Zhang, Zhang, Yi, Tu, Yang ICLR 2024, oral)
#   https://arxiv.org/abs/2310.09656  https://github.com/amazon-science/tabsyn
#
# Idea: instead of running diffusion in feature space (TabDDPM) or
# sampling z from a BGM (VAE-BGM), TabSyn (a) fits a VAE to map x -> z
# with mild KL regularisation so the latent is approximately Gaussian,
# (b) trains a DDPM in that latent space, (c) samples z via diffusion
# and decodes. Reported -86%/-67% error vs strongest baselines on the
# six standard tabular benchmarks (column-wise / pair-wise correlation).
#
# This implementation is a NUMERIC-ONLY adaptation -- our 15 Sentinel-2
# vegetation indices are continuous so we drop TabSyn's tokeniser and
# Transformer (intended for mixed types). Components:
#   - TabSynVAE: MLP encoder/decoder, deterministic decoder (MSE rec
#                loss, KL on encoder), Gaussian prior. Light KL weight
#                (Zhang 2024 sec 3.2: "embeddings should retain mutual
#                info while staying close to N(0,I) for diffusion to
#                exploit").
#   - LatentDDPM: DDPM (cosine schedule) on encoded z.
#   - Reuses TabDDPMDenoiser MLP from section above (operates on latent
#     dim now instead of feature dim).
#
# Notes for our crop pipeline:
#   - Train VAE on z-scored x; encode all train -> mu(x); train DDPM on
#     these mu's; sample mu* via reverse DDPM; decode to x_hat; inverse
#     z-score; quantile-match.
#   - Latent dim chosen by class_vae_capacity() so small classes get a
#     compact latent (matches v9 small-class fix).
# =====================================================================


class TabSynVAE(nn.Module):
    """Plain Gaussian VAE with KL annealing -- numeric-only TabSyn core."""

    def __init__(self, n_features, latent_dim=8, hidden_size=128, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.latent_dim = latent_dim
        self.enc = nn.Sequential(
            nn.Linear(n_features, hidden_size), nn.BatchNorm1d(hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.BatchNorm1d(hidden_size), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(hidden_size, latent_dim)
        self.fc_logvar = nn.Linear(hidden_size, latent_dim)
        self.dec = nn.Sequential(
            nn.Linear(latent_dim, hidden_size), nn.BatchNorm1d(hidden_size), nn.ReLU(),
            nn.Linear(hidden_size, hidden_size), nn.BatchNorm1d(hidden_size), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, n_features),
        )

    def encode(self, x):
        h = self.enc(x)
        mu = self.fc_mu(h)
        log_var = torch.clamp(self.fc_logvar(h), min=-5.0, max=2.0)
        return mu, log_var

    def reparam(self, mu, log_var):
        eps = torch.randn_like(mu)
        return mu + torch.exp(0.5 * log_var) * eps

    def decode(self, z):
        return self.dec(z)

    def forward(self, x):
        mu, log_var = self.encode(x)
        z = self.reparam(mu, log_var)
        x_rec = self.decode(z)
        return x_rec, mu, log_var, z

    def fit(self, X_train, X_val, n_epochs=200, batch_size=4096, lr=1e-3,
            kl_warmup_epochs=30, beta_max=0.1, weight_decay=1e-4,
            grad_clip=5.0, patience=30, device=DEVICE, verbose=True):
        """KL beta capped at 0.1 (TabSyn paper sec 3.2 -- low KL keeps
        z close to N(0,I) without crushing rec quality)."""
        self.to(device)
        opt = torch.optim.Adam(self.parameters(), lr=lr, weight_decay=weight_decay)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode='min', factor=0.5, patience=10, min_lr=1e-5)
        Xt = torch.as_tensor(X_train, dtype=torch.float32, device=device)
        Xv = torch.as_tensor(X_val, dtype=torch.float32, device=device)
        n_train = Xt.shape[0]
        best_val = float('inf')
        best_state = None
        bad = 0
        history = {'train': [], 'val': [], 'rec': [], 'kl': []}
        for epoch in range(n_epochs):
            beta = min(beta_max, beta_max * (epoch + 1) / max(1, kl_warmup_epochs))
            self.train()
            perm = torch.randperm(n_train, device=device)
            ep_loss = 0.0
            n_b = (n_train + batch_size - 1) // batch_size
            for b in range(n_b):
                idx = perm[b * batch_size:(b + 1) * batch_size]
                if idx.shape[0] < 2:
                    continue
                xb = Xt[idx]
                x_rec, mu, log_var, _ = self(xb)
                rec = F.mse_loss(x_rec, xb)
                kl = -0.5 * torch.mean(1 + log_var - mu.pow(2) - log_var.exp())
                loss = rec + beta * kl
                opt.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.parameters(), grad_clip)
                opt.step()
                ep_loss += loss.item()
            ep_loss /= max(1, n_b)
            self.eval()
            with torch.no_grad():
                xv_rec, mu_v, lv_v, _ = self(Xv)
                rec_v = F.mse_loss(xv_rec, Xv).item()
                kl_v = (-0.5 * torch.mean(1 + lv_v - mu_v.pow(2) - lv_v.exp())).item()
                val_loss = rec_v + beta_max * kl_v
            history['train'].append(ep_loss)
            history['val'].append(val_loss)
            history['rec'].append(rec_v)
            history['kl'].append(kl_v)
            if val_loss < best_val - 1e-5:
                best_val = val_loss
                best_state = {k: v.detach().cpu().clone()
                              for k, v in self.state_dict().items()}
                bad = 0
            else:
                bad += 1
            sched.step(val_loss)
            if verbose and epoch % 20 == 0:
                print(f'    VAE ep {epoch:3d} | train={ep_loss:.4f} | '
                      f'val={val_loss:.4f} (rec={rec_v:.4f}, kl={kl_v:.4f}) | beta={beta:.3f}')
            if bad >= patience:
                if verbose:
                    print(f'    VAE early stop ep {epoch} (best={best_val:.4f})')
                break
        if best_state is not None:
            self.load_state_dict(best_state)
        history['best_val'] = best_val
        del Xt, Xv
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        return history

    @torch.no_grad()
    def encode_all(self, X, batch=8192, device=DEVICE):
        self.eval()
        self.to(device)
        Xt = torch.as_tensor(X, dtype=torch.float32, device=device)
        chunks = []
        for i in range(0, Xt.shape[0], batch):
            mu, _ = self.encode(Xt[i:i + batch])
            chunks.append(mu.cpu().numpy())
        del Xt
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        return np.vstack(chunks).astype(np.float32)

    @torch.no_grad()
    def decode_all(self, Z, batch=8192, device=DEVICE):
        self.eval()
        self.to(device)
        Zt = torch.as_tensor(Z, dtype=torch.float32, device=device)
        chunks = []
        for i in range(0, Zt.shape[0], batch):
            chunks.append(self.decode(Zt[i:i + batch]).cpu().numpy())
        del Zt
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        return np.vstack(chunks).astype(np.float32)


class LatentDDPM(nn.Module):
    """DDPM in latent z-space (TabSyn step 2). Reuses TabDDPMDenoiser MLP
    -- now operating on latent_dim instead of feature_dim."""

    def __init__(self, latent_dim, T=1000, hidden=256, time_emb_dim=128,
                 n_blocks=3, dropout=0.1):
        super().__init__()
        self.latent_dim = latent_dim
        self.T = T
        self.denoiser = TabDDPMDenoiser(
            n_features=latent_dim, hidden=hidden, time_emb_dim=time_emb_dim,
            n_blocks=n_blocks, dropout=dropout,
        )
        betas = cosine_beta_schedule(T)
        alphas = 1.0 - betas
        alpha_bars = torch.cumprod(alphas, dim=0)
        self.register_buffer('betas', betas)
        self.register_buffer('alphas', alphas)
        self.register_buffer('alpha_bars', alpha_bars)
        self.register_buffer('sqrt_alpha_bars', torch.sqrt(alpha_bars))
        self.register_buffer('sqrt_one_minus_alpha_bars', torch.sqrt(1.0 - alpha_bars))

    def q_sample(self, z0, t, noise):
        sa = self.sqrt_alpha_bars[t].unsqueeze(-1)
        som = self.sqrt_one_minus_alpha_bars[t].unsqueeze(-1)
        return sa * z0 + som * noise

    def loss(self, z0):
        b = z0.shape[0]
        t = torch.randint(0, self.T, (b,), device=z0.device)
        noise = torch.randn_like(z0)
        z_t = self.q_sample(z0, t, noise)
        pred = self.denoiser(z_t, t)
        return F.mse_loss(pred, noise)

    @torch.no_grad()
    def sample(self, n_samples, device=None, batch=4096):
        if device is None:
            device = next(self.parameters()).device
        self.eval()
        out = []
        remaining = n_samples
        while remaining > 0:
            b = min(batch, remaining)
            x = torch.randn(b, self.latent_dim, device=device)
            for t in reversed(range(self.T)):
                t_b = torch.full((b,), t, device=device, dtype=torch.long)
                pred_noise = self.denoiser(x, t_b)
                alpha_t = self.alphas[t]
                alpha_bar_t = self.alpha_bars[t]
                beta_t = self.betas[t]
                coef = beta_t / torch.sqrt(1.0 - alpha_bar_t)
                mean = (1.0 / torch.sqrt(alpha_t)) * (x - coef * pred_noise)
                if t > 0:
                    sigma = torch.sqrt(beta_t)
                    x = mean + sigma * torch.randn_like(x)
                else:
                    x = mean
            out.append(x.cpu().numpy())
            remaining -= b
        return np.vstack(out).astype(np.float32)

    def fit(self, Z_train, Z_val, n_epochs=300, batch_size=4096, lr=1e-3,
            weight_decay=1e-4, grad_clip=5.0, patience=40, device=DEVICE,
            verbose=True):
        self.to(device)
        opt = torch.optim.Adam(self.parameters(), lr=lr, weight_decay=weight_decay)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
            opt, mode='min', factor=0.5, patience=15, min_lr=1e-5)
        Zt = torch.as_tensor(Z_train, dtype=torch.float32, device=device)
        Zv = torch.as_tensor(Z_val, dtype=torch.float32, device=device)
        n_train = Zt.shape[0]
        history = {'train': [], 'val': []}
        best_val = float('inf')
        best_state = None
        bad = 0
        for epoch in range(n_epochs):
            self.train()
            perm = torch.randperm(n_train, device=device)
            ep_loss = 0.0
            n_b = (n_train + batch_size - 1) // batch_size
            for b in range(n_b):
                idx = perm[b * batch_size:(b + 1) * batch_size]
                if idx.shape[0] < 2:
                    continue
                z = Zt[idx]
                loss = self.loss(z)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.parameters(), grad_clip)
                opt.step()
                ep_loss += loss.item()
            ep_loss /= max(1, n_b)
            self.eval()
            with torch.no_grad():
                val_loss = self.loss(Zv).item()
            history['train'].append(ep_loss)
            history['val'].append(val_loss)
            sched.step(val_loss)
            if val_loss < best_val - 1e-5:
                best_val = val_loss
                best_state = {k: v.detach().cpu().clone()
                              for k, v in self.state_dict().items()}
                bad = 0
            else:
                bad += 1
            if verbose and epoch % 25 == 0:
                print(f'    DDPM ep {epoch:3d} | train={ep_loss:.4f} | val={val_loss:.4f}')
            if bad >= patience:
                if verbose:
                    print(f'    DDPM early stop ep {epoch} (best={best_val:.4f})')
                break
        if best_state is not None:
            self.load_state_dict(best_state)
        history['best_val'] = best_val
        del Zt, Zv
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        return history


print('TabSyn (TabSynVAE + LatentDDPM) defined.')


## 17. TabSyn Per-Class Generation + RF Eval


In [ ]:
# =====================================================================
# TabSyn PER-CLASS GENERATION (v19 -- cached + SMOTE fallback)
#   Pipeline per class:
#     1. Train TabSynVAE on z-scored real (Gaussian VAE, KL=0.1)
#     2. Encode all real train -> Z_real (mu only -- deterministic)
#     3. Train LatentDDPM on Z_real
#     4. Sample Z_synth via DDPM, decode to X_synth, inverse z-score
#     5. Quantile-match marginals to real (same as VAE-BGM/TabDDPM)
#
#   v19 (2026-05-03): SMOTE fallback when real_count < min_real_for_vae.
#     See result18_analysis.md sec 5.1. TabSyn trained on 1.6K samples
#     hurt 2420 F1 (-0.022 vs real-only) -- two-stage VAE+DDPM compounds
#     under-determined error. Now tiny classes route to SMOTE.
# =====================================================================
TABSYN_CONFIG = {
    'beta_max': 0.1,
    'vae_epochs': 200,
    'vae_lr': 1e-3,
    'vae_batch': 4096,
    'vae_kl_warmup': 30,
    'vae_patience': 30,
    'ddpm_T': 1000,
    'ddpm_hidden': 256,
    'ddpm_n_blocks': 3,
    'ddpm_time_emb': 128,
    'ddpm_dropout': 0.1,
    'ddpm_epochs': 300,
    'ddpm_lr': 1e-3,
    'ddpm_batch': 4096,
    'ddpm_patience': 40,
    'ddpm_sample_batch': 4096,
    'winsorize_pct': CONFIG['winsorize_pct'],
    'val_split': CONFIG['val_split'],
    'quantile_match': CONFIG['quantile_match'],
}


def tabsyn_capacity(n_real):
    """Latent dim + hidden scale with N (Bishop guideline, same as v9)."""
    if n_real < CONFIG['min_real_for_vae']:
        return {'latent_dim': 4, 'hidden_size': 64,
                'ddpm_hidden': 128, 'ddpm_n_blocks': 2}
    if n_real < CONFIG['anchored_threshold']:
        return {'latent_dim': 4, 'hidden_size': 64,
                'ddpm_hidden': 192, 'ddpm_n_blocks': 3}
    return {'latent_dim': 8, 'hidden_size': 128,
            'ddpm_hidden': 256, 'ddpm_n_blocks': 3}


synthetic_paths_tabsyn = {}
tabsyn_histories = {}

for cls_idx, (label, n_gen) in enumerate(classes_to_generate.items()):
    real_count = int(class_counts[label])
    use_smote_fb = real_count < CONFIG['min_real_for_vae']
    cap = tabsyn_capacity(real_count)
    gen_cfg = {'method': 'TabSyn', 'config': TABSYN_CONFIG, 'capacity': cap,
               'fallback': 'SMOTE' if use_smote_fb else None}
    real_class_df = raw_df[raw_df['label'] == label]
    key = cache_key('tabsyn', label, real_class_df, feature_cols, gen_cfg, n_gen)
    hit = cache_load('tabsyn', label, key)

    method_str = 'SMOTE-fb' if use_smote_fb else 'TabSyn'
    print(f'\n[TabSyn] Class {label} ({cls_idx + 1}/{len(classes_to_generate)}) '
          f'-- {real_count:,} real, gen {n_gen:,}  '
          f'method={method_str}  '
          f'cap=(z={cap["latent_dim"]}, h={cap["hidden_size"]}, '
          f'ddpm_h={cap["ddpm_hidden"]}, ddpm_b={cap["ddpm_n_blocks"]})  '
          f'cache: tabsyn/{label}_{key}')
    if hit is not None:
        print(f'  CACHE HIT -> {hit}')
        synthetic_paths_tabsyn[label] = hit
        continue

    # v19: SMOTE fallback for tiny classes -- no VAE/DDPM training.
    if use_smote_fb:
        t0 = time.time()
        cls_real = (raw_df[raw_df['label'] == label][feature_cols]
                    .dropna().to_numpy(dtype=np.float32))
        syn_arr = smote_generate(
            cls_real, n_gen,
            k=CONFIG['smote_k_neighbors'], random_state=42)
        gen_df = pd.DataFrame(syn_arr, columns=feature_cols)
        gen_df['label'] = label
        syn_path = cache_save(
            'tabsyn', label, key, gen_df,
            {'gen_config': gen_cfg, 'n_gen': int(n_gen),
             'fallback': 'SMOTE', 'real_count': real_count})
        synthetic_paths_tabsyn[label] = syn_path
        tabsyn_histories[label] = None
        print(f'  v19 SMOTE fallback (N={real_count:,} '
              f'< min_real_for_vae={CONFIG["min_real_for_vae"]:,}) '
              f'-> {len(gen_df):,} samples in '
              f'{time.time() - t0:.1f}s -> {syn_path}')
        del cls_real, syn_arr, gen_df
        gc.collect()
        continue

    t0 = time.time()
    X_tr, _, X_va, _, norm_stats = prepare_class_data(
        raw_df, label, feature_cols,
        max_samples=CONFIG['max_train_samples'],
        val_split=TABSYN_CONFIG['val_split'],
        winsorize_pct=TABSYN_CONFIG['winsorize_pct'],
    )
    print(f'  Data: {X_tr.shape[0]:,} train, {X_va.shape[0]:,} val')

    torch.manual_seed(42)
    np.random.seed(42)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(42)

    vae = TabSynVAE(
        n_features=n_features,
        latent_dim=cap['latent_dim'],
        hidden_size=cap['hidden_size'],
    )
    print('  Stage 1/2: train Gaussian VAE...')
    h_vae = vae.fit(
        X_tr, X_va,
        n_epochs=TABSYN_CONFIG['vae_epochs'],
        batch_size=min(TABSYN_CONFIG['vae_batch'], max(64, len(X_tr) // 4)),
        lr=TABSYN_CONFIG['vae_lr'],
        kl_warmup_epochs=TABSYN_CONFIG['vae_kl_warmup'],
        beta_max=TABSYN_CONFIG['beta_max'],
        patience=TABSYN_CONFIG['vae_patience'],
        device=DEVICE, verbose=True,
    )

    Z_tr = vae.encode_all(X_tr, device=DEVICE)
    Z_va = vae.encode_all(X_va, device=DEVICE)
    print(f'  Encoded: Z_tr {Z_tr.shape}, Z_va {Z_va.shape}, '
          f'mean={Z_tr.mean():.3f} std={Z_tr.std():.3f}')

    ddpm = LatentDDPM(
        latent_dim=cap['latent_dim'],
        T=TABSYN_CONFIG['ddpm_T'],
        hidden=cap['ddpm_hidden'],
        n_blocks=cap['ddpm_n_blocks'],
        time_emb_dim=TABSYN_CONFIG['ddpm_time_emb'],
        dropout=TABSYN_CONFIG['ddpm_dropout'],
    )
    print('  Stage 2/2: train Latent DDPM...')
    h_ddpm = ddpm.fit(
        Z_tr, Z_va,
        n_epochs=TABSYN_CONFIG['ddpm_epochs'],
        batch_size=min(TABSYN_CONFIG['ddpm_batch'], max(64, len(Z_tr) // 4)),
        lr=TABSYN_CONFIG['ddpm_lr'],
        patience=TABSYN_CONFIG['ddpm_patience'],
        device=DEVICE, verbose=True,
    )
    tabsyn_histories[label] = {'vae': h_vae, 'ddpm': h_ddpm}
    print(f'  Trained in {time.time() - t0:.1f}s. '
          f'VAE best val={h_vae["best_val"]:.4f}, '
          f'DDPM best val={h_ddpm["best_val"]:.4f}')

    print(f'  Sampling {n_gen:,} via reverse DDPM (T={TABSYN_CONFIG["ddpm_T"]}) ...')
    t1 = time.time()
    Z_syn = ddpm.sample(n_gen, device=DEVICE, batch=TABSYN_CONFIG['ddpm_sample_batch'])
    X_syn = vae.decode_all(Z_syn, device=DEVICE)
    print(f'  Sampled+decoded in {time.time() - t1:.1f}s')

    gen_df = denormalize(X_syn, feature_cols, norm_stats)
    gen_df['label'] = label
    if TABSYN_CONFIG['quantile_match']:
        real_only = raw_df[raw_df['label'] == label][feature_cols].dropna()
        gen_df = match_real_marginals(gen_df, real_only, feature_cols)
        gen_df['label'] = label
        del real_only

    syn_path = cache_save('tabsyn', label, key, gen_df,
                          {'gen_config': gen_cfg, 'n_gen': int(n_gen),
                           'vae_best_val': h_vae['best_val'],
                           'ddpm_best_val': h_ddpm['best_val']})
    synthetic_paths_tabsyn[label] = syn_path
    print(f'  Saved -> {syn_path}')
    del vae, ddpm, X_tr, X_va, Z_tr, Z_va, Z_syn, X_syn, gen_df
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

print(f'\nTabSyn done. {len(synthetic_paths_tabsyn)} classes.')


In [ ]:
# Augmented RF -- Real + TabSyn synthetic
if synthetic_paths_tabsyn:
    y_pred_s = _build_aug_rf_eval(synthetic_paths_tabsyn, 'TabSyn')
    print('\n--- AUGMENTED: Real + TabSyn Synthetic ---')
    print(classification_report(y_test, y_pred_s, zero_division=0))
    _print_paper_metrics(y_test, y_pred_s, 'Test set -- Real + TabSyn')
else:
    print('Skipping TabSyn augmented RF (no synthetic_paths_tabsyn).')


## 18. Final 5-Way RF Comparison (All Generators)


In [ ]:
# =====================================================================
# Final head-to-head -- ALL generators evaluated against the SAME paper-
# exact RF baseline. Auto-skips methods whose y_pred_* is undefined.
# =====================================================================
predictions = [('Real Only', y_pred_real)]
if 'y_pred_aug' in dir():
    predictions.append(('Real+VAE-BGM', y_pred_aug))
if 'y_pred_t' in dir():
    predictions.append(('Real+TVAE', y_pred_t))
if 'y_pred_d' in dir():
    predictions.append(('Real+TabDDPM', y_pred_d))
if 'y_pred_s' in dir():
    predictions.append(('Real+TabSyn', y_pred_s))

labels_sorted = sorted(class_counts.keys())
n_methods = len(predictions)

f1_per_method = {
    name: f1_score(y_test, yp, labels=labels_sorted, average=None, zero_division=0)
    for name, yp in predictions
}

cmp_data = {'Class': labels_sorted, 'Count': [class_counts[l] for l in labels_sorted]}
for name in f1_per_method:
    cmp_data[f'F1_{name}'] = np.round(f1_per_method[name], 4)
cmp_data['Generated'] = ['Yes' if l in classes_to_generate else 'No' for l in labels_sorted]
cmp = pd.DataFrame(cmp_data)
print('Per-class F1 -- all methods:')
print(cmp.to_string(index=False))

print(f'\n{"Method":<14s} {"Acc":>8s} {"Kappa":>7s} {"Macro F1":>10s} '
      f'{"Weighted F1":>12s} {"Bal Acc":>9s}')
print('-' * 64)
summary_rows = []
for name, yp in predictions:
    a = accuracy_score(y_test, yp)
    k = cohen_kappa_score(y_test, yp)
    m = f1_score(y_test, yp, average='macro', zero_division=0)
    w = f1_score(y_test, yp, average='weighted', zero_division=0)
    b = balanced_accuracy_score(y_test, yp)
    print(f'{name:<14s} {a:>8.4f} {k:>7.4f} {m:>10.4f} {w:>12.4f} {b:>9.4f}')
    summary_rows.append({'Method': name, 'Accuracy': a, 'Kappa': k,
                         'Macro_F1': m, 'Weighted_F1': w, 'Balanced_Acc': b})

# Best per metric
print('\nBest per metric:')
sum_df = pd.DataFrame(summary_rows)
for metric in ['Accuracy', 'Kappa', 'Macro_F1', 'Weighted_F1', 'Balanced_Acc']:
    best = sum_df.loc[sum_df[metric].idxmax()]
    print(f'  {metric:<14s} {best["Method"]:<15s} {best[metric]:.4f}')

fig, ax = plt.subplots(figsize=(16, 5))
x = np.arange(len(labels_sorted))
w_bar = 0.8 / n_methods
colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple', 'goldenrod']
for i, (name, _) in enumerate(predictions):
    offset = (i - (n_methods - 1) / 2) * w_bar
    ax.bar(x + offset, f1_per_method[name], w_bar, label=name, color=colors[i % len(colors)])
ax.set_xticks(x)
ax.set_xticklabels(labels_sorted, rotation=45)
ax.set_xlabel('Crop Class')
ax.set_ylabel('F1')
ax.set_title(f'Per-class F1: {n_methods}-way RF comparison (Real / VAE-BGM / TVAE / TabDDPM / TabSyn)')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'f1_{n_methods}way_comparison.png'), dpi=150)
plt.show()

cmp.to_csv(os.path.join(OUTPUT_DIR, f'f1_{n_methods}way_comparison.csv'), index=False)
sum_df.to_csv(os.path.join(OUTPUT_DIR, f'summary_metrics_{n_methods}way.csv'), index=False)
print(f'\nSaved: f1_{n_methods}way_comparison.csv  +  png  +  summary_metrics_{n_methods}way.csv')
